# Daily Full Funnel Performance Query - All Volume

For all compass and net call steps, gets throughput/breakage/queue rates, and the volume that made it through.

Additionally, have expected conversion and financial plan information. 

Aggregating all data

In [0]:
!pip install tqdm
!pip install -U python-docx
!pip install openai
!pip install openpyxl
!pip install dotenv
!pip install slack_sdk

# Creating KPI JSON

### Data Aggregation

In [0]:
from typing import Optional, Union, Sequence
from pyspark.sql import DataFrame
from datetime import date, timedelta

def _compute_yday_and_prior4_dates(today: Optional[date] = None):
    if today is None:
        today = date.today()
    yday = today - timedelta(days=1)
    dates = [yday - timedelta(days=7 * i) for i in range(0, 5)]
    return [d.strftime("%Y-%m-%d") for d in dates]


# Columns actually used by compute_kpis()
NEEDED_VCALLS_COLS = [
    "call_date","call_id","company_id","call_direction","spanish_ind",
    "ib_gross_ind","ib_queue_ind","ibs_net_ind","last_ivr_selection_name",
    # used for marketing_bucket + site_serp derivations
    "ivr_split_name","web_session_id",
]

NEEDED_CCF_COLS = [
    "call_id",
    "call_connected_timestamp","compass_start_time","moving_switching_asked_time",
    "movingSwitchingExtracted",
    "return_caller_confirmation_extracted","return_caller_dtmf_boolean","return_caller_confirmation_response",
    "web_address_confirmation_extracted","web_address_confirmation_dtmf_boolean","web_address_confirmation_response",
    "web_zip_confirm_extracted","web_zip_confirmation_dtmf_boolean","web_zip_confirmation_response",
    "web_dob_name_collected_time",
    "zip_code_collected_time","zip_code_confirm_extracted",
    "ss_address_confirmed_time","web_address_confirmed_time","return_caller_confirmed_time",
    "ivrrouting","ercotMatch",
    "name_collected_time","return_caller_path","dob_collected_time",
    "credit_api_start_time","credit_response_status","isCreditHit",
    "first_workflow_leg_id",
    "dead_air_bot","home_business_response",
    "initial_ivr_vf","texas_nontexas_extracted","home_business_extracted",
]

NEEDED_AC_COLS = [
    "call_id",
    "ib_contact_calls",
    "credit_calls_ind",
    "order_count",
    "passed_credit_call_ind",
    "allconnect_transition_ind",
    "partner_name", 
]

NEEDED_O_COLS = [
    "call_id",
    "gcv_v2",
]


def get_data(
    marketing_buckets: Optional[Union[str, Sequence[str]]] = None,
    site_serp: Optional[Union[str, Sequence[str]]] = None,
    company_id: Optional[int] = 25,
    call_direction: Optional[str] = "INBOUND",
    restrict_to_yday_and_prior4: bool = True,
    create_temp_view: bool = True,
    print_summary: bool = True,
) -> DataFrame:
    """
    Faster get_data(): only selects columns required by compute_kpis()
    (plus columns needed to derive marketing_bucket/site_serp).

    Returns a wide calls_full DataFrame and optionally creates TEMP VIEW calls_full.
    """

    # ---------------------------
    # Normalize & validate filters (same behavior as before)
    # ---------------------------
    allowed_buckets = {
        "Natural","Brand-Partner","Generic","Aggregator","Competitor","Utility","PMax","NRG","Other Bucket"
    }

    def _as_list(x):
        if x is None:
            return None
        if isinstance(x, str):
            return [x]
        return list(x)

    mb_list = _as_list(marketing_buckets)
    if mb_list is not None:
        norm_map = {
            "brandpartner": "Brand-Partner",
            "brand-partner": "Brand-Partner",
            "brand_partner": "Brand-Partner",
            "pmax": "PMax",
            "nrg": "NRG",
            "other": "Other Bucket",
            "other bucket": "Other Bucket",
            "utility": "Utility",
            "natural": "Natural",
            "generic": "Generic",
            "aggregator": "Aggregator",
            "competitor": "Competitor",
        }
        mb_list = [norm_map.get(b.strip().lower(), b) for b in mb_list]
        unknown = [b for b in mb_list if b not in allowed_buckets]
        if unknown:
            raise ValueError(f"Unknown marketing_buckets: {unknown}. Allowed: {sorted(allowed_buckets)}")

    ss_list = _as_list(site_serp)
    if ss_list is not None:
        norm_ss = {"site": "Site", "serp": "SERP"}
        ss_list = [norm_ss.get(s.strip().lower(), s) for s in ss_list]
        unknown = [s for s in ss_list if s not in {"Site", "SERP"}]
        if unknown:
            raise ValueError("site_serp must be 'Site', 'SERP', or a list of those values.")

    # ---------------------------
    # Source tables
    # ---------------------------
    base_table = "energy_prod.energy.v_calls"
    ccf_table  = "ai_products_prod.energy.compass_call_fct"
    ac_table   = "energy_prod.energy.v_agent_calls"
    o_table    = "energy_prod.energy.v_orders"

    # ---------------------------
    # CASE expressions (unchanged)
    # ---------------------------
    marketing_bucket_case = """
      CASE
        WHEN c.ivr_split_name IN ('natural_marketingbucket','natural_marketingbucket_serp') THEN 'Natural'
        WHEN c.ivr_split_name IN ('brandpartner_marketingbucket_serp','brandpartner_marketingbucket') THEN 'Brand-Partner'
        WHEN c.ivr_split_name IN ('generic_marketingbucket_serp','generic_marketingbucket') THEN 'Generic'
        WHEN c.ivr_split_name IN ('aggregator_marketingbucket_serp','aggregator_marketingbucket') THEN 'Aggregator'
        WHEN c.ivr_split_name IN ('competitor_marketingbucket_serp','competitor_marketingbucket') THEN 'Competitor'
        WHEN c.ivr_split_name IN ('dereg_utility_check_serp','dereg_utility_check') THEN 'Utility'
        WHEN c.ivr_split_name IN ('pmax_marketingbucket_serp','pmax_marketingbucket') THEN 'PMax'
        WHEN c.ivr_split_name IN ('nrg_bucket_serp','nrg_bucket') THEN 'NRG'
        ELSE 'Other Bucket'
      END
    """

    site_serp_case = """
      CASE
        WHEN c.web_session_id IS NOT NULL THEN 'Site'
        ELSE 'SERP'
      END
    """

    # ---------------------------
    # WHERE (base)
    # ---------------------------
    base_where_clauses = []
    if company_id is not None:
        base_where_clauses.append(f"c.company_id = {int(company_id)}")
    if call_direction is not None:
        safe_call_dir = str(call_direction).replace("'", "''")
        base_where_clauses.append(f"UPPER(c.call_direction) = UPPER('{safe_call_dir}')")

    pulled_dates = None
    if restrict_to_yday_and_prior4:
        pulled_dates = _compute_yday_and_prior4_dates()
        dates_sql = ", ".join([f"'{d}'" for d in pulled_dates])
        base_where_clauses.append(f"c.call_date IN ({dates_sql})")

    base_where_sql = "\n        AND ".join(base_where_clauses) if base_where_clauses else "1=1"

    # post filters (marketing bucket / site_serp)
    post_filters = []
    if mb_list:
        mb_sql = ", ".join([f"'{b}'" for b in mb_list])
        post_filters.append(f"marketing_bucket IN ({mb_sql})")
    if ss_list:
        ss_sql = ", ".join([f"'{s}'" for s in ss_list])
        post_filters.append(f"site_serp IN ({ss_sql})")

    post_where_sql = ""
    if post_filters:
        post_where_sql = "WHERE " + " AND ".join(post_filters)

    # ---------------------------
    # SELECT lists (only required cols)
    # ---------------------------
    # v_calls needed + derived fields
    base_select_cols = ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS])

    # ccf/ac/o needed, with prefixes to match compute_kpis expectations
    ccf_select_cols = ",\n      ".join(
        [f"ccf.`{col}` AS `ccf_{col}`" for col in NEEDED_CCF_COLS if col != "call_id"]
    )
    ac_select_cols = ",\n      ".join(
        [f"ac.`{col}` AS `ac_{col}`" for col in NEEDED_AC_COLS if col != "call_id"]
    )
    o_select_cols = ",\n      ".join(
        [f"o.`{col}` AS `o_{col}`" for col in NEEDED_O_COLS if col != "call_id"]
    )

    # final projection order
    final_select_parts = [
        # base calls (including ivr_split_name/web_session_id needed for derivations)
        ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS]),
        "c.marketing_bucket",
        "c.site_serp",
        ccf_select_cols,
        ac_select_cols,
        o_select_cols,
    ]
    final_select_sql = ",\n      ".join([p for p in final_select_parts if p.strip()])

    calls_full_sql = f"""
    WITH base_calls AS (
      SELECT
        {base_select_cols},
        {marketing_bucket_case} AS marketing_bucket,
        {site_serp_case} AS site_serp
      FROM {base_table} c
      WHERE {base_where_sql}
    ),
    filtered_calls AS (
      SELECT *
      FROM base_calls
      {post_where_sql}
    )
    SELECT
      {final_select_sql}
    FROM filtered_calls c
    LEFT JOIN {ccf_table} ccf
      ON c.call_id = ccf.call_id
    LEFT JOIN {ac_table} ac
      ON c.call_id = ac.call_id
    LEFT JOIN {o_table} o
      ON c.call_id = o.call_id
    """

    df = spark.sql(calls_full_sql)

    if create_temp_view:
        df.createOrReplaceTempView("calls_full")

    if print_summary:
        if pulled_dates is not None:
            print(f"get_data(): pulled call_date IN {pulled_dates}")
        else:
            print("get_data(): no call_date restriction applied")
        print(f"get_data(): rows = {df.count()}")
        if create_temp_view:
            print("get_data(): created TEMP VIEW calls_full")

    return df


### Full Funnel Query

In [0]:
from typing import Optional
from pyspark.sql import DataFrame

def compute_kpis(
    calls_df: Optional[DataFrame] = None,
    source_view: str = "calls_full",
    create_temp_view: bool = True,
    debug_print_sql: bool = True,
) -> DataFrame:
    """
    Serverless-friendly compute_kpis where the full KPI SQL is embedded in the function.

    - If calls_df is provided and create_temp_view=True, it will createOrReplaceTempView(source_view).
    - The internal SQL references the given source_view and creates a temp view named out_view.
    - Returns a DataFrame from SELECT * FROM out_view.
    """

    # If caller provided a DataFrame, register it as the source view
    if calls_df is not None and create_temp_view:
        calls_df.createOrReplaceTempView(source_view)

    # Build a safe output view name (unique-ish)
    safe_view = (
        source_view.replace("`", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
        .replace("|", "_")
        .replace("/", "_")
    )
    out_view = f"full_funnel_kpis__{safe_view}"

    # Embedded SQL (no date-filtering here; computation expects the source_view to already be limited to relevant dates)
    full_funnel_kpis_sql = f"""
CREATE OR REPLACE TEMP VIEW {out_view} AS
WITH params AS (
  SELECT
    date_sub(current_date(), 1)            AS yday,
    dayofweek(date_sub(current_date(), 1)) AS yday_dow,
    date_format(date_sub(current_date(), 1), 'yyyy-MM-dd') AS yday_label
),

base AS (
  SELECT
    cf.call_date,
    cf.call_id,

    -- v_calls (no prefix in calls_full)
    cf.company_id,
    cf.call_direction,
    cf.spanish_ind,
    cf.ib_gross_ind,
    cf.ib_queue_ind,
    cf.ibs_net_ind,

    -- v_agent_calls (ac_)
    cf.ac_ib_contact_calls,
    cf.ac_credit_calls_ind,
    cf.ac_order_count,
    cf.ac_passed_credit_call_ind,
    cf.ac_allconnect_transition_ind,
    cf.ac_partner_name,

    -- v_orders (o_)
    cf.o_gcv_v2,

    -- Entered Compass + key step flags
    CASE
      WHEN try_cast(cf.ccf_call_connected_timestamp AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_compass_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS entered_compass,

    CASE
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_asked,

    CASE
      WHEN cf.ccf_movingSwitchingExtracted IS NOT NULL
           AND cf.ccf_movingSwitchingExtracted NOT IN ('0','null') THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_extracted,

    CASE
      WHEN try_cast(cf.ccf_zip_code_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_zip_code_confirm_extracted IN ('Yes','yes') THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS any_zip_collected,

    CASE
      WHEN try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes') THEN 1
      ELSE 0
    END AS any_address_collected,

    CASE
      WHEN cf.ccf_ivrrouting = 'default'
       AND cf.ccf_ercotMatch = 'true'
       AND (
            try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL
         OR try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL
         OR cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
       )
      THEN 1 ELSE 0
    END AS address_collected_matched,

    CASE
      WHEN try_cast(cf.ccf_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS name_confirmed,

    CASE
      WHEN try_cast(cf.ccf_dob_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS dob_collected,

    CASE
      WHEN try_cast(cf.ccf_credit_api_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ucc_api_call,

    CASE
      WHEN cf.ccf_credit_response_status = 'Successful' THEN 1
      ELSE 0
    END AS ucc_api_call_successful,

    CASE
      WHEN cf.ccf_isCreditHit = 'true' THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
           AND cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS ucc_credit_hit,

    CASE
      WHEN cf.ccf_first_workflow_leg_id IS NOT NULL THEN 1
      ELSE 0
    END AS compass_completion,

    -- Bot flags
    CASE WHEN cf.ccf_dead_air_bot = '1' THEN 1 ELSE 0 END AS deadair_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE
        '(^|[^0-9])' ||
        '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
        '(' ||
          '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{3}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{4}}' ||
        ')' ||
        '([^0-9]|$)'
      THEN 1 ELSE 0
    END AS phone_number_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]'
      THEN 1 ELSE 0
    END AS press_number_bot_traffic,

    CASE
      WHEN (cf.ccf_dead_air_bot = '1')
        OR (cf.ccf_home_business_response RLIKE
          '(^|[^0-9])' ||
          '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
          '(' ||
            '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{3}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{4}}' ||
          ')' ||
          '([^0-9]|$)')
        OR (cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]')
      THEN 1 ELSE 0
    END AS any_bot_call,

    -- IVR completion proxy
    CASE
      WHEN cf.last_ivr_selection_name IN ('TX','TX-1') THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1 AND cf.ccf_texas_nontexas_extracted = 'Texas' THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1
           AND cf.ccf_home_business_extracted = 'residential'
           AND cf.call_date < DATE '2025-09-24' THEN 1
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ivr_completed_call,

    -- STEP REACH denominators
    CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END AS reached_entered_compass,
    CASE WHEN entered_compass = 1 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS reached_moving_switching,
    CASE WHEN entered_compass = 1 AND moving_switching_extracted = 1 THEN 1 ELSE 0 END AS reached_zip_collection,
    CASE WHEN entered_compass = 1 AND any_zip_collected = 1 THEN 1 ELSE 0 END AS reached_address_collection,
    CASE WHEN entered_compass = 1 AND any_address_collected = 1 THEN 1 ELSE 0 END AS reached_ercot_check,
    CASE WHEN entered_compass = 1 AND address_collected_matched = 1 THEN 1 ELSE 0 END AS reached_name_collection,
    CASE WHEN entered_compass = 1 AND name_confirmed = 1 THEN 1 ELSE 0 END AS reached_dob_collection,
    CASE WHEN entered_compass = 1 AND dob_collected = 1 THEN 1 ELSE 0 END AS reached_credit_check,

    -- QUEUE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 1 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS q_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 0 THEN 1 ELSE 0 END AS q_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 1 AND any_zip_collected = 0 THEN 1 ELSE 0 END AS q_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS q_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS q_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS q_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS q_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS q_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS q_with_credit_hit,

    -- BREAKAGE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 0 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS b_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 0 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS b_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 1
              AND try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL
              AND any_zip_collected = 0
         THEN 1 ELSE 0 END AS b_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS b_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS b_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 0 AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS b_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS b_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS b_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 0 AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS b_with_credit_hit

  FROM {source_view} cf
  CROSS JOIN params p
  WHERE cf.call_date >= date_sub(p.yday, 60)
    AND cf.call_date <= p.yday
    AND dayofweek(cf.call_date) = p.yday_dow
    AND cf.spanish_ind = 0
    AND cf.company_id = 25
    AND cf.call_direction IN ('Inbound','INBOUND')
),

prior4_dates AS (
  SELECT call_date
  FROM (
    SELECT
      call_date,
      dense_rank() OVER (ORDER BY call_date DESC) AS dr
    FROM (SELECT DISTINCT call_date FROM base)
    CROSS JOIN params p
    WHERE call_date < p.yday
  ) t
  WHERE dr <= 4
),

scoped AS (
  SELECT
    b.*,
    CASE
      WHEN b.call_date = p.yday THEN 'yesterday'
      WHEN b.call_date IN (SELECT call_date FROM prior4_dates) THEN 'prior4'
      ELSE 'ignore'
    END AS period
  FROM base b
  CROSS JOIN params p
),

daily AS (
  SELECT
    call_date,
    period,

    -- Core volumes
    SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END) AS gross_calls,

    SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END) AS entered_compass_calls,
    try_divide(
      SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS entered_compass_rate,

    try_divide(
      SUM(compass_completion),
      NULLIF(SUM(entered_compass), 0)
    ) AS compass_completion_rate,

    -- Bot KPIs (Compass-scoped)
    try_divide(
      SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END), 0)
    ) AS any_bot_call_rate_compass,

    ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
      - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
    ) AS non_bot_compass_calls,

    -- Queue metrics
    SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END) AS queue_calls,
    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_gross_rate,

    try_divide(
      SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_ivr_rate,

    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 AND ibs_net_ind = 0 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS abandonment_rate,

    -- Net + downstream rates
    SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END) AS net_calls,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_ib_contact_calls = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS contact_rate,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_credit_calls_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS credit_rate,

    try_divide(
      SUM(CASE WHEN ac_credit_calls_ind = 1 AND ac_passed_credit_call_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_credit_calls_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS passed_credit_rate,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 AND ac_passed_credit_call_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_passed_credit_call_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS passed_credit_conversion_rate,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 AND ac_credit_calls_ind = 1 AND (ac_passed_credit_call_ind = 0 OR ac_passed_credit_call_ind IS NULL) THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_credit_calls_ind = 1 AND (ac_passed_credit_call_ind = 0 OR ac_passed_credit_call_ind IS NULL) THEN 1 ELSE 0 END), 0)
    ) AS failed_credit_conversion_rate,

    /* -------------------------
       ALLCONNECT TRANSFERS
       Definition: allconnect_transfer_ind / orders
       ------------------------- */
    try_divide(
      SUM(CASE WHEN ac_allconnect_transition_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS allconnect_transfer_rate,

    /* -------------------------
       TXU / TEE ORDER MIX
       Definition: % of orders by partner
       - TXU partner_name: 'TXU Energy'
       - TEE partner_name: 'TriEagle Energy'
       ------------------------- */
    try_divide(
      SUM(CASE WHEN ac_order_count = 1 AND ac_partner_name = 'TXU Energy' THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS txu_order_mix,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 AND ac_partner_name = 'TriEagle Energy' THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS tee_order_mix, 

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS net_conversion_rate,

    SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END) AS orders,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS gross_conversion_rate,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(
        ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
          - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
        ),
        0
      )
    ) AS transactional_conversion_rate,

    -- UCC operational rates
    try_divide(SUM(ucc_api_call),            NULLIF(SUM(entered_compass), 0)) AS ucc_api_call_rate,
    try_divide(SUM(ucc_api_call_successful), NULLIF(SUM(ucc_api_call), 0))    AS ucc_api_success_given_call_rate,
    try_divide(SUM(ucc_credit_hit),          NULLIF(SUM(ucc_api_call_successful), 0)) AS ucc_hit_given_success_rate,

    -- Step loss rates
    try_divide(SUM(q_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS q_rate_initial_question,
    try_divide(SUM(b_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS b_rate_initial_question,

    try_divide(SUM(q_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS q_rate_moving_switching,
    try_divide(SUM(b_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS b_rate_moving_switching,

    try_divide(SUM(q_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS q_rate_zip_collection,
    try_divide(SUM(b_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS b_rate_zip_collection,

    try_divide(SUM(q_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS q_rate_address_collection,
    try_divide(SUM(b_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS b_rate_address_collection,

    try_divide(SUM(q_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS q_rate_ercot_check,
    try_divide(SUM(b_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS b_rate_ercot_check,

    try_divide(SUM(q_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS q_rate_name_collection,
    try_divide(SUM(b_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS b_rate_name_collection,

    try_divide(SUM(q_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS q_rate_dob_collection,
    try_divide(SUM(b_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS b_rate_dob_collection,

    try_divide(SUM(q_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_credit_check_no_hit,
    try_divide(SUM(b_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_credit_check_no_hit,

    try_divide(SUM(q_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_with_credit_hit,
    try_divide(SUM(b_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_with_credit_hit,

    -- Step counts (unchanged)
    SUM(reached_entered_compass) AS reached_entered_compass_calls,
    SUM(reached_moving_switching) AS reached_moving_switching_calls,
    SUM(reached_zip_collection) AS reached_zip_collection_calls,
    SUM(reached_address_collection) AS reached_address_collection_calls,
    SUM(reached_ercot_check) AS reached_ercot_check_calls,
    SUM(reached_name_collection) AS reached_name_collection_calls,
    SUM(reached_dob_collection) AS reached_dob_collection_calls,
    SUM(reached_credit_check) AS reached_credit_check_calls,

    SUM(q_at_initial_question) AS q_at_initial_question_calls,
    SUM(b_at_initial_question) AS b_at_initial_question_calls,
    SUM(q_at_moving_switching) AS q_at_moving_switching_calls,
    SUM(b_at_moving_switching) AS b_at_moving_switching_calls,
    SUM(q_at_zip_collection) AS q_at_zip_collection_calls,
    SUM(b_at_zip_collection) AS b_at_zip_collection_calls,
    SUM(q_at_address_collection) AS q_at_address_collection_calls,
    SUM(b_at_address_collection) AS b_at_address_collection_calls,
    SUM(q_at_ercot_check) AS q_at_ercot_check_calls,
    SUM(b_at_ercot_check) AS b_at_ercot_check_calls,
    SUM(q_at_name_collection) AS q_at_name_collection_calls,
    SUM(b_at_name_collection) AS b_at_name_collection_calls,
    SUM(q_at_dob_collection) AS q_at_dob_collection_calls,
    SUM(b_at_dob_collection) AS b_at_dob_collection_calls,
    SUM(q_at_credit_check_no_hit) AS q_at_credit_check_no_hit_calls,
    SUM(b_at_credit_check_no_hit) AS b_at_credit_check_no_hit_calls,
    SUM(q_with_credit_hit) AS q_with_credit_hit_calls,
    SUM(b_with_credit_hit) AS b_with_credit_hit_calls,

    SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END) AS ivr_completed_calls,

    SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END) AS total_revenue,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_order,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_net_call,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_gross_call

  FROM scoped
  WHERE period IN ('yesterday','prior4')
  GROUP BY call_date, period
),

period_agg AS (
  SELECT
    MAX(p.yday_label) AS yday_label,

    AVG(CASE WHEN period='prior4' THEN CAST(gross_calls AS DOUBLE) END) AS p4_gross_calls,
    AVG(CASE WHEN period='prior4' THEN entered_compass_rate END) AS p4_entered_compass_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(entered_compass_calls AS DOUBLE) END) AS p4_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN compass_completion_rate END) AS p4_compass_completion_rate,

    AVG(CASE WHEN period='prior4' THEN any_bot_call_rate_compass END) AS p4_any_bot_call_rate_compass,
    AVG(CASE WHEN period='prior4' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS p4_non_bot_compass_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_gross_rate END) AS p4_queue_to_gross_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(queue_calls AS DOUBLE) END) AS p4_queue_calls,
    AVG(CASE WHEN period='prior4' THEN abandonment_rate END) AS p4_abandonment_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(net_calls AS DOUBLE) END) AS p4_net_calls,

    AVG(CASE WHEN period='prior4' THEN contact_rate END) AS p4_contact_rate,
    AVG(CASE WHEN period='prior4' THEN credit_rate END) AS p4_credit_rate,
    AVG(CASE WHEN period='prior4' THEN passed_credit_rate END)              AS p4_passed_credit_rate,
    AVG(CASE WHEN period='prior4' THEN passed_credit_conversion_rate END)   AS p4_passed_credit_conversion_rate,
    AVG(CASE WHEN period='prior4' THEN failed_credit_conversion_rate END)   AS p4_failed_credit_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN allconnect_transfer_rate END)        AS p4_allconnect_transfer_rate,

    AVG(CASE WHEN period='prior4' THEN txu_order_mix END)                   AS p4_txu_order_mix,
    AVG(CASE WHEN period='prior4' THEN tee_order_mix END)                   AS p4_tee_order_mix,

    AVG(CASE WHEN period='prior4' THEN net_conversion_rate END) AS p4_net_conversion_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(orders AS DOUBLE) END) AS p4_orders,
    AVG(CASE WHEN period='prior4' THEN gross_conversion_rate END) AS p4_gross_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN ucc_api_call_rate END) AS p4_ucc_api_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_api_success_given_call_rate END) AS p4_ucc_api_success_given_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_hit_given_success_rate END) AS p4_ucc_hit_given_success_rate,

    AVG(CASE WHEN period='prior4' THEN q_rate_initial_question END) AS p4_q_rate_initial_question,
    AVG(CASE WHEN period='prior4' THEN b_rate_initial_question END) AS p4_b_rate_initial_question,

    AVG(CASE WHEN period='prior4' THEN q_rate_moving_switching END) AS p4_q_rate_moving_switching,
    AVG(CASE WHEN period='prior4' THEN b_rate_moving_switching END) AS p4_b_rate_moving_switching,

    AVG(CASE WHEN period='prior4' THEN q_rate_zip_collection END) AS p4_q_rate_zip_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_zip_collection END) AS p4_b_rate_zip_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_address_collection END) AS p4_q_rate_address_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_address_collection END) AS p4_b_rate_address_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_ercot_check END) AS p4_q_rate_ercot_check,
    AVG(CASE WHEN period='prior4' THEN b_rate_ercot_check END) AS p4_b_rate_ercot_check,

    AVG(CASE WHEN period='prior4' THEN q_rate_name_collection END) AS p4_q_rate_name_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_name_collection END) AS p4_b_rate_name_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_dob_collection END) AS p4_q_rate_dob_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_dob_collection END) AS p4_b_rate_dob_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_credit_check_no_hit END) AS p4_q_rate_credit_check_no_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_credit_check_no_hit END) AS p4_b_rate_credit_check_no_hit,

    AVG(CASE WHEN period='prior4' THEN q_rate_with_credit_hit END) AS p4_q_rate_with_credit_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_with_credit_hit END) AS p4_b_rate_with_credit_hit,

    -- step counts prior4
    AVG(CASE WHEN period='prior4' THEN CAST(reached_entered_compass_calls AS DOUBLE) END)      AS p4_reached_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_moving_switching_calls AS DOUBLE) END)     AS p4_reached_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_zip_collection_calls AS DOUBLE) END)       AS p4_reached_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_address_collection_calls AS DOUBLE) END)   AS p4_reached_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_ercot_check_calls AS DOUBLE) END)          AS p4_reached_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_name_collection_calls AS DOUBLE) END)      AS p4_reached_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_dob_collection_calls AS DOUBLE) END)       AS p4_reached_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_credit_check_calls AS DOUBLE) END)         AS p4_reached_credit_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_initial_question_calls AS DOUBLE) END)        AS p4_q_at_initial_question_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_initial_question_calls AS DOUBLE) END)        AS p4_b_at_initial_question_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_moving_switching_calls AS DOUBLE) END)        AS p4_q_at_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_moving_switching_calls AS DOUBLE) END)        AS p4_b_at_moving_switching_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_zip_collection_calls AS DOUBLE) END)          AS p4_q_at_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_zip_collection_calls AS DOUBLE) END)          AS p4_b_at_zip_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_address_collection_calls AS DOUBLE) END)      AS p4_q_at_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_address_collection_calls AS DOUBLE) END)      AS p4_b_at_address_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_ercot_check_calls AS DOUBLE) END)             AS p4_q_at_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_ercot_check_calls AS DOUBLE) END)             AS p4_b_at_ercot_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_name_collection_calls AS DOUBLE) END)         AS p4_q_at_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_name_collection_calls AS DOUBLE) END)         AS p4_b_at_name_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_dob_collection_calls AS DOUBLE) END)          AS p4_q_at_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_dob_collection_calls AS DOUBLE) END)          AS p4_b_at_dob_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_q_at_credit_check_no_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_b_at_credit_check_no_hit_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_with_credit_hit_calls AS DOUBLE) END)            AS p4_q_with_credit_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_with_credit_hit_calls AS DOUBLE) END)            AS p4_b_with_credit_hit_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_ivr_rate END) AS p4_queue_to_ivr_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(ivr_completed_calls AS DOUBLE) END) AS p4_ivr_completed_calls,

    AVG(CASE WHEN period='prior4' THEN transactional_conversion_rate END) AS p4_transactional_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN CAST(total_revenue AS DOUBLE) END) AS p4_total_revenue,
    AVG(CASE WHEN period='prior4' THEN revenue_per_order END) AS p4_revenue_per_order,
    AVG(CASE WHEN period='prior4' THEN revenue_per_net_call END) AS p4_revenue_per_net_call,
    AVG(CASE WHEN period='prior4' THEN revenue_per_gross_call END) AS p4_revenue_per_gross_call,

    -- yesterday values
    MAX(CASE WHEN period='yesterday' THEN CAST(gross_calls AS DOUBLE) END) AS y_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN entered_compass_rate END) AS y_entered_compass_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(entered_compass_calls AS DOUBLE) END) AS y_entered_compass_calls,
    MAX(CASE WHEN period='yesterday' THEN compass_completion_rate END) AS y_compass_completion_rate,

    MAX(CASE WHEN period='yesterday' THEN any_bot_call_rate_compass END) AS y_any_bot_call_rate_compass,
    MAX(CASE WHEN period='yesterday' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS y_non_bot_compass_calls,

    MAX(CASE WHEN period='yesterday' THEN queue_to_gross_rate END) AS y_queue_to_gross_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(queue_calls AS DOUBLE) END) AS y_queue_calls,
    MAX(CASE WHEN period='yesterday' THEN abandonment_rate END) AS y_abandonment_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(net_calls AS DOUBLE) END) AS y_net_calls,

    MAX(CASE WHEN period='yesterday' THEN contact_rate END) AS y_contact_rate,
    MAX(CASE WHEN period='yesterday' THEN credit_rate END) AS y_credit_rate,
    MAX(CASE WHEN period='yesterday' THEN passed_credit_rate END)            AS y_passed_credit_rate,
    MAX(CASE WHEN period='yesterday' THEN passed_credit_conversion_rate END) AS y_passed_credit_conversion_rate,
    MAX(CASE WHEN period='yesterday' THEN failed_credit_conversion_rate END) AS y_failed_credit_conversion_rate,

    MAX(CASE WHEN period='yesterday' THEN allconnect_transfer_rate END)      AS y_allconnect_transfer_rate,

    MAX(CASE WHEN period='yesterday' THEN txu_order_mix END)                 AS y_txu_order_mix,
    MAX(CASE WHEN period='yesterday' THEN tee_order_mix END)                 AS y_tee_order_mix,
    MAX(CASE WHEN period='yesterday' THEN net_conversion_rate END) AS y_net_conversion_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(orders AS DOUBLE) END) AS y_orders,
    MAX(CASE WHEN period='yesterday' THEN gross_conversion_rate END) AS y_gross_conversion_rate,

    MAX(CASE WHEN period='yesterday' THEN ucc_api_call_rate END) AS y_ucc_api_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_api_success_given_call_rate END) AS y_ucc_api_success_given_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_hit_given_success_rate END) AS y_ucc_hit_given_success_rate,

    MAX(CASE WHEN period='yesterday' THEN q_rate_initial_question END) AS y_q_rate_initial_question,
    MAX(CASE WHEN period='yesterday' THEN b_rate_initial_question END) AS y_b_rate_initial_question,

    MAX(CASE WHEN period='yesterday' THEN q_rate_moving_switching END) AS y_q_rate_moving_switching,
    MAX(CASE WHEN period='yesterday' THEN b_rate_moving_switching END) AS y_b_rate_moving_switching,

    MAX(CASE WHEN period='yesterday' THEN q_rate_zip_collection END) AS y_q_rate_zip_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_zip_collection END) AS y_b_rate_zip_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_address_collection END) AS y_q_rate_address_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_address_collection END) AS y_b_rate_address_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_ercot_check END) AS y_q_rate_ercot_check,
    MAX(CASE WHEN period='yesterday' THEN b_rate_ercot_check END) AS y_b_rate_ercot_check,

    MAX(CASE WHEN period='yesterday' THEN q_rate_name_collection END) AS y_q_rate_name_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_name_collection END) AS y_b_rate_name_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_dob_collection END) AS y_q_rate_dob_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_dob_collection END) AS y_b_rate_dob_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_credit_check_no_hit END) AS y_q_rate_credit_check_no_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_credit_check_no_hit END) AS y_b_rate_credit_check_no_hit,

    MAX(CASE WHEN period='yesterday' THEN q_rate_with_credit_hit END) AS y_q_rate_with_credit_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_with_credit_hit END) AS y_b_rate_with_credit_hit,

    MAX(CASE WHEN period='yesterday' THEN CAST(total_revenue AS DOUBLE) END) AS y_total_revenue,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_order END)            AS y_revenue_per_order,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_net_call END)         AS y_revenue_per_net_call,
    MAX(CASE WHEN period='yesterday' THEN revenue_per_gross_call END)       AS y_revenue_per_gross_call,

    MAX(CASE WHEN period='yesterday' THEN queue_to_ivr_rate END) AS y_queue_to_ivr_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(ivr_completed_calls AS DOUBLE) END) AS y_ivr_completed_calls,
    MAX(CASE WHEN period='yesterday' THEN transactional_conversion_rate END) AS y_transactional_conversion_rate

  FROM daily
  CROSS JOIN params p
),

final AS (

  /* -------------------------
     VOLUME + ENTRY
     ------------------------- */

  SELECT  1 AS sort_key, 'Gross Calls' AS KPI,
          p4_gross_calls AS P4WA, y_gross_calls AS y_value,
          try_divide(y_gross_calls - p4_gross_calls, NULLIF(p4_gross_calls,0)) AS Delta,
          (y_gross_calls - p4_gross_calls) AS Call_Count_Delta
  FROM period_agg

  UNION ALL SELECT  2, 'Entered Compass Calls (Count)',
          p4_entered_compass_calls, y_entered_compass_calls,
          try_divide(y_entered_compass_calls - p4_entered_compass_calls, NULLIF(p4_entered_compass_calls,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  3, 'Entered Compass Rate',
          p4_entered_compass_rate, y_entered_compass_rate,
          try_divide(y_entered_compass_rate - p4_entered_compass_rate, NULLIF(p4_entered_compass_rate,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg


  /* -------------------------
     BOT
     ------------------------- */

  UNION ALL SELECT  4, 'Any Bot Call Rate (Compass)',
          p4_any_bot_call_rate_compass, y_any_bot_call_rate_compass,
          try_divide(y_any_bot_call_rate_compass - p4_any_bot_call_rate_compass, NULLIF(p4_any_bot_call_rate_compass,0)),
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  5, 'Non-Bot Compass Calls (Count)',
          p4_non_bot_compass_calls, y_non_bot_compass_calls,
          try_divide(y_non_bot_compass_calls - p4_non_bot_compass_calls, NULLIF(p4_non_bot_compass_calls,0)),
          (y_non_bot_compass_calls - p4_non_bot_compass_calls)
  FROM period_agg


  /* -------------------------
     STEP-LEVEL QUEUE RATES
     ------------------------- */

  UNION ALL SELECT  30, 'Queue Rate – Initial Question',
          p4_q_rate_initial_question, y_q_rate_initial_question,
          try_divide(y_q_rate_initial_question - p4_q_rate_initial_question, NULLIF(p4_q_rate_initial_question,0)),
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  31, 'Queue Rate – Moving / Switching',
          p4_q_rate_moving_switching, y_q_rate_moving_switching,
          try_divide(y_q_rate_moving_switching - p4_q_rate_moving_switching, NULLIF(p4_q_rate_moving_switching,0)),
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  32, 'Queue Rate – ZIP Collection',
          p4_q_rate_zip_collection, y_q_rate_zip_collection,
          try_divide(y_q_rate_zip_collection - p4_q_rate_zip_collection, NULLIF(p4_q_rate_zip_collection,0)),
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  33, 'Queue Rate – Address Collection',
          p4_q_rate_address_collection, y_q_rate_address_collection,
          try_divide(y_q_rate_address_collection - p4_q_rate_address_collection, NULLIF(p4_q_rate_address_collection,0)),
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  34, 'Queue Rate – ERCOT Check',
          p4_q_rate_ercot_check, y_q_rate_ercot_check,
          try_divide(y_q_rate_ercot_check - p4_q_rate_ercot_check, NULLIF(p4_q_rate_ercot_check,0)),
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  35, 'Queue Rate – Name Collection',
          p4_q_rate_name_collection, y_q_rate_name_collection,
          try_divide(y_q_rate_name_collection - p4_q_rate_name_collection, NULLIF(p4_q_rate_name_collection,0)),
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  36, 'Queue Rate – DOB Collection',
          p4_q_rate_dob_collection, y_q_rate_dob_collection,
          try_divide(y_q_rate_dob_collection - p4_q_rate_dob_collection, NULLIF(p4_q_rate_dob_collection,0)),
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  37, 'Queue Rate – Credit Check (No Hit)',
          p4_q_rate_credit_check_no_hit, y_q_rate_credit_check_no_hit,
          try_divide(y_q_rate_credit_check_no_hit - p4_q_rate_credit_check_no_hit, NULLIF(p4_q_rate_credit_check_no_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  38, 'Queue Rate – Credit Check (Hit)',
          p4_q_rate_with_credit_hit, y_q_rate_with_credit_hit,
          try_divide(y_q_rate_with_credit_hit - p4_q_rate_with_credit_hit, NULLIF(p4_q_rate_with_credit_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg


  /* -------------------------
     STEP-LEVEL BREAKAGE RATES
     ------------------------- */

  UNION ALL SELECT  40, 'Breakage Rate – Initial Question',
          p4_b_rate_initial_question, y_b_rate_initial_question,
          try_divide(y_b_rate_initial_question - p4_b_rate_initial_question, NULLIF(p4_b_rate_initial_question,0)),
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  41, 'Breakage Rate – Moving / Switching',
          p4_b_rate_moving_switching, y_b_rate_moving_switching,
          try_divide(y_b_rate_moving_switching - p4_b_rate_moving_switching, NULLIF(p4_b_rate_moving_switching,0)),
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  42, 'Breakage Rate – ZIP Collection',
          p4_b_rate_zip_collection, y_b_rate_zip_collection,
          try_divide(y_b_rate_zip_collection - p4_b_rate_zip_collection, NULLIF(p4_b_rate_zip_collection,0)),
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  43, 'Breakage Rate – Address Collection',
          p4_b_rate_address_collection, y_b_rate_address_collection,
          try_divide(y_b_rate_address_collection - p4_b_rate_address_collection, NULLIF(p4_b_rate_address_collection,0)),
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  44, 'Breakage Rate – ERCOT Check',
          p4_b_rate_ercot_check, y_b_rate_ercot_check,
          try_divide(y_b_rate_ercot_check - p4_b_rate_ercot_check, NULLIF(p4_b_rate_ercot_check,0)),
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  45, 'Breakage Rate – Name Collection',
          p4_b_rate_name_collection, y_b_rate_name_collection,
          try_divide(y_b_rate_name_collection - p4_b_rate_name_collection, NULLIF(p4_b_rate_name_collection,0)),
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  46, 'Breakage Rate – DOB Collection',
          p4_b_rate_dob_collection, y_b_rate_dob_collection,
          try_divide(y_b_rate_dob_collection - p4_b_rate_dob_collection, NULLIF(p4_b_rate_dob_collection,0)),
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  47, 'Breakage Rate – Credit Check (No Hit)',
          p4_b_rate_credit_check_no_hit, y_b_rate_credit_check_no_hit,
          try_divide(y_b_rate_credit_check_no_hit - p4_b_rate_credit_check_no_hit, NULLIF(p4_b_rate_credit_check_no_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  48, 'Breakage Rate – Credit Check (Hit)',
          p4_b_rate_with_credit_hit, y_b_rate_with_credit_hit,
          try_divide(y_b_rate_with_credit_hit - p4_b_rate_with_credit_hit, NULLIF(p4_b_rate_with_credit_hit,0)),
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg


  /* -------------------------
     IVR COMPLETION (place before Queue / Abandon per request)
     ------------------------- */

  UNION ALL SELECT  49, 'Passed Initial Question',
          p4_ivr_completed_calls, y_ivr_completed_calls,
          try_divide(y_ivr_completed_calls - p4_ivr_completed_calls, NULLIF(p4_ivr_completed_calls,0)),
          (y_ivr_completed_calls - p4_ivr_completed_calls)
  FROM period_agg

  UNION ALL SELECT  49.1, 'Queue to Initial Question Passed Rate',
          p4_queue_to_ivr_rate, y_queue_to_ivr_rate,
          try_divide(y_queue_to_ivr_rate - p4_queue_to_ivr_rate, NULLIF(p4_queue_to_ivr_rate,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg


  /* -------------------------
     QUEUE / ABANDON
     ------------------------- */

  UNION ALL SELECT  50, 'Queue Calls',
          p4_queue_calls, y_queue_calls,
          try_divide(y_queue_calls - p4_queue_calls, NULLIF(p4_queue_calls,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg

  UNION ALL SELECT  51, 'Queue to Gross Rate',
          p4_queue_to_gross_rate, y_queue_to_gross_rate,
          try_divide(y_queue_to_gross_rate - p4_queue_to_gross_rate, NULLIF(p4_queue_to_gross_rate,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

  UNION ALL SELECT  52, 'Abandonment Rate - Hung Up in Queue',
          p4_abandonment_rate, y_abandonment_rate,
          try_divide(y_abandonment_rate - p4_abandonment_rate, NULLIF(p4_abandonment_rate,0)),
          (y_queue_calls - p4_queue_calls)
  FROM period_agg


  /* -------------------------
     NET / SALES
     ------------------------- */

  UNION ALL SELECT  60, 'Net Calls',
          p4_net_calls, y_net_calls,
          try_divide(y_net_calls - p4_net_calls, NULLIF(p4_net_calls,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  61, 'Contact Calls/Net Calls',
          p4_contact_rate, y_contact_rate,
          try_divide(y_contact_rate - p4_contact_rate, NULLIF(p4_contact_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  62, 'Credit Run/Net Calls',
          p4_credit_rate, y_credit_rate,
          try_divide(y_credit_rate - p4_credit_rate, NULLIF(p4_credit_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg
  /* -------------------------
     CREDIT PASS + SPLIT CONVERSION
     ------------------------- */

  UNION ALL SELECT  62.1, 'Passed Credit Rate (Passed / Reached Credit)',
          p4_passed_credit_rate, y_passed_credit_rate,
          try_divide(y_passed_credit_rate - p4_passed_credit_rate, NULLIF(p4_passed_credit_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  63.1, 'Passed Credit Conversion Rate',
          p4_passed_credit_conversion_rate, y_passed_credit_conversion_rate,
          try_divide(y_passed_credit_conversion_rate - p4_passed_credit_conversion_rate, NULLIF(p4_passed_credit_conversion_rate,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  63.2, 'Failed Credit Conversion Rate',
          p4_failed_credit_conversion_rate, y_failed_credit_conversion_rate,
          try_divide(y_failed_credit_conversion_rate - p4_failed_credit_conversion_rate, NULLIF(p4_failed_credit_conversion_rate,0)),
          (y_orders - p4_orders)
  FROM period_agg


  /* -------------------------
     ALLCONNECT TRANSFERS
     ------------------------- */

  UNION ALL SELECT  66.0, 'Allconnect Transfer Rate (Transfers / Orders)',
          p4_allconnect_transfer_rate, y_allconnect_transfer_rate,
          try_divide(y_allconnect_transfer_rate - p4_allconnect_transfer_rate, NULLIF(p4_allconnect_transfer_rate,0)),
          (y_orders - p4_orders)
  FROM period_agg


  /* -------------------------
     PARTNER MIX (OF ORDERS)
     ------------------------- */

  UNION ALL SELECT  66.5, 'TXU Order Mix (% of Orders)',
          p4_txu_order_mix, y_txu_order_mix,
          try_divide(y_txu_order_mix - p4_txu_order_mix, NULLIF(p4_txu_order_mix,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  66.6, 'TEE Order Mix (% of Orders)',
          p4_tee_order_mix, y_tee_order_mix,
          try_divide(y_tee_order_mix - p4_tee_order_mix, NULLIF(p4_tee_order_mix,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  63, 'Net Conversion Rate',
          p4_net_conversion_rate, y_net_conversion_rate,
          try_divide(y_net_conversion_rate - p4_net_conversion_rate, NULLIF(p4_net_conversion_rate,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  64, 'Orders',
          p4_orders, y_orders,
          try_divide(y_orders - p4_orders, NULLIF(p4_orders,0)),
          (y_orders - p4_orders)
  FROM period_agg

  -- transactional conversion rate BEFORE gross conversion rate (per request)
  UNION ALL SELECT  64.5, 'Non-Bot Compass Call Conversion Rate',
          p4_transactional_conversion_rate, y_transactional_conversion_rate,
          try_divide(y_transactional_conversion_rate - p4_transactional_conversion_rate, NULLIF(p4_transactional_conversion_rate,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  65, 'Gross Conversion Rate',
          p4_gross_conversion_rate, y_gross_conversion_rate,
          try_divide(y_gross_conversion_rate - p4_gross_conversion_rate, NULLIF(p4_gross_conversion_rate,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg


  /* -------------------------
     REVENUE (at the end)
     ------------------------- */

  UNION ALL SELECT  90, 'Total Revenue',
          p4_total_revenue, y_total_revenue,
          try_divide(y_total_revenue - p4_total_revenue, NULLIF(p4_total_revenue,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  91, 'Revenue per Order',
          p4_revenue_per_order, y_revenue_per_order,
          try_divide(y_revenue_per_order - p4_revenue_per_order, NULLIF(p4_revenue_per_order,0)),
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  92, 'Revenue per Net Call',
          p4_revenue_per_net_call, y_revenue_per_net_call,
          try_divide(y_revenue_per_net_call - p4_revenue_per_net_call, NULLIF(p4_revenue_per_net_call,0)),
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  93, 'Revenue per Gross Call',
          p4_revenue_per_gross_call, y_revenue_per_gross_call,
          try_divide(y_revenue_per_gross_call - p4_revenue_per_gross_call, NULLIF(p4_revenue_per_gross_call,0)),
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

)

SELECT
  sort_key,
  KPI,
  P4WA,
  y_value AS `Yesterday`,
  Delta,
  Call_Count_Delta
FROM final
ORDER BY sort_key
;

"""

    # Optionally preview the SQL (first N chars)
    if debug_print_sql:
        print(f"--- compute_kpis() will create/replace temp view: {out_view} ---")
        # print a short preview so notebook output remains readable
        print(full_funnel_kpis_sql[:3000])
        print("--- end SQL preview ---\n")

    try:
        # Execute the embedded SQL which must create the {out_view} temp view
        spark.sql(full_funnel_kpis_sql)

        # Return the results
        return spark.sql(f"SELECT * FROM {out_view}")

    except Exception as e:
        print("ERROR running compute_kpis():", type(e).__name__, str(e))
        try:
            print("\n--- SQL (first 8000 chars) ---")
            print(full_funnel_kpis_sql[:8000])
            print("\n--- SQL (last 8000 chars) ---")
            print(full_funnel_kpis_sql[-8000:])
        except Exception:
            pass
        raise


### Mix Shift Query

In [0]:
from typing import Optional
from pyspark.sql import DataFrame

def compute_mix_shifts(
    calls_df: Optional[DataFrame] = None,
    source_view: str = "calls_full",
    create_temp_view: bool = False,
    debug_print_sql: bool = False,
) -> DataFrame:
    """
    Compatible with get_data() output.

    Assumptions (per get_data):
      - Data already contains ONLY yday + prior4 (5 dates) when restrict_to_yday_and_prior4=True
      - Data already has derived columns:
          - marketing_bucket  (e.g., Natural, Brand-Partner, ... Other Bucket)
          - site_serp         ('Site' or 'SERP')

    Output rows are two separate “universes”:
      - row_type = 'CATEGORY'  with row_name = marketing_bucket
      - row_type = 'SITE_SERP' with row_name = site_serp

    Columns (9 total):
      Gross_P4_Mix, Gross_Yday_Mix, Gross_Mix_Pct_Change,
      Net_P4_Mix,   Net_Yday_Mix,   Net_Mix_Pct_Change,
      Orders_P4_Mix,Orders_Yday_Mix,Orders_Mix_Pct_Change

    Mix denominators are separate per universe (CATEGORY sums to 1, SITE_SERP sums to 1).
    """

    if calls_df is not None and create_temp_view:
        calls_df.createOrReplaceTempView(source_view)

    sql = f"""
WITH params AS (
  SELECT date_sub(current_date(), 1) AS yday
),

base AS (
  SELECT
    cf.call_date,
    cf.marketing_bucket,
    cf.site_serp,
    cf.ib_gross_ind,
    cf.ibs_net_ind,
    cf.ac_order_count
  FROM {source_view} cf
),

scoped AS (
  SELECT
    b.*,
    CASE
      WHEN b.call_date = p.yday THEN 'yesterday'
      ELSE 'prior4'
    END AS period
  FROM base b
  CROSS JOIN params p
),

/* -----------------------------
   CATEGORY universe
   ----------------------------- */
daily_category AS (
  SELECT
    call_date,
    period,
    marketing_bucket AS row_name,
    SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END) AS gross_calls,
    SUM(CASE WHEN ibs_net_ind  = 1 THEN 1 ELSE 0 END) AS net_calls,
    SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END) AS orders
  FROM scoped
  GROUP BY call_date, period, marketing_bucket
),

daily_category_totals AS (
  SELECT
    call_date,
    period,
    SUM(gross_calls) AS total_gross_calls,
    SUM(net_calls)   AS total_net_calls,
    SUM(orders)      AS total_orders
  FROM daily_category
  GROUP BY call_date, period
),

period_category_totals AS (
  SELECT
    'CATEGORY' AS row_type,
    MAX(CASE WHEN period='yesterday' THEN total_gross_calls END) AS y_total_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN total_net_calls   END) AS y_total_net_calls,
    MAX(CASE WHEN period='yesterday' THEN total_orders      END) AS y_total_orders,

    AVG(CASE WHEN period='prior4' THEN CAST(total_gross_calls AS DOUBLE) END) AS p4_total_gross_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(total_net_calls   AS DOUBLE) END) AS p4_total_net_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(total_orders      AS DOUBLE) END) AS p4_total_orders
  FROM daily_category_totals
),

period_category_dim AS (
  SELECT
    'CATEGORY' AS row_type,
    row_name,

    MAX(CASE WHEN period='yesterday' THEN CAST(gross_calls AS DOUBLE) END) AS y_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN CAST(net_calls   AS DOUBLE) END) AS y_net_calls,
    MAX(CASE WHEN period='yesterday' THEN CAST(orders      AS DOUBLE) END) AS y_orders,

    AVG(CASE WHEN period='prior4' THEN CAST(gross_calls AS DOUBLE) END) AS p4_gross_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(net_calls   AS DOUBLE) END) AS p4_net_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(orders      AS DOUBLE) END) AS p4_orders
  FROM daily_category
  GROUP BY row_name
),

category_out AS (
  SELECT
    d.row_type,
    d.row_name,

    try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)) AS Gross_P4_Mix,
    try_divide(d.y_gross_calls,  NULLIF(t.y_total_gross_calls, 0))  AS Gross_Yday_Mix,
    try_divide(
      try_divide(d.y_gross_calls, NULLIF(t.y_total_gross_calls, 0))
      - try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)),
      NULLIF(try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)), 0)
    ) AS Gross_Mix_Pct_Change,

    try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)) AS Net_P4_Mix,
    try_divide(d.y_net_calls,  NULLIF(t.y_total_net_calls, 0))  AS Net_Yday_Mix,
    try_divide(
      try_divide(d.y_net_calls, NULLIF(t.y_total_net_calls, 0))
      - try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)),
      NULLIF(try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)), 0)
    ) AS Net_Mix_Pct_Change,

    try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)) AS Orders_P4_Mix,
    try_divide(d.y_orders,  NULLIF(t.y_total_orders, 0))  AS Orders_Yday_Mix,
    try_divide(
      try_divide(d.y_orders, NULLIF(t.y_total_orders, 0))
      - try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)),
      NULLIF(try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)), 0)
    ) AS Orders_Mix_Pct_Change
  FROM period_category_dim d
  CROSS JOIN period_category_totals t
),

/* -----------------------------
   SITE/SERP universe
   ----------------------------- */
daily_siteserp AS (
  SELECT
    call_date,
    period,
    site_serp AS row_name,
    SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END) AS gross_calls,
    SUM(CASE WHEN ibs_net_ind  = 1 THEN 1 ELSE 0 END) AS net_calls,
    SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END) AS orders
  FROM scoped
  GROUP BY call_date, period, site_serp
),

daily_siteserp_totals AS (
  SELECT
    call_date,
    period,
    SUM(gross_calls) AS total_gross_calls,
    SUM(net_calls)   AS total_net_calls,
    SUM(orders)      AS total_orders
  FROM daily_siteserp
  GROUP BY call_date, period
),

period_siteserp_totals AS (
  SELECT
    'SITE_SERP' AS row_type,
    MAX(CASE WHEN period='yesterday' THEN total_gross_calls END) AS y_total_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN total_net_calls   END) AS y_total_net_calls,
    MAX(CASE WHEN period='yesterday' THEN total_orders      END) AS y_total_orders,

    AVG(CASE WHEN period='prior4' THEN CAST(total_gross_calls AS DOUBLE) END) AS p4_total_gross_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(total_net_calls   AS DOUBLE) END) AS p4_total_net_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(total_orders      AS DOUBLE) END) AS p4_total_orders
  FROM daily_siteserp_totals
),

period_siteserp_dim AS (
  SELECT
    'SITE_SERP' AS row_type,
    row_name,

    MAX(CASE WHEN period='yesterday' THEN CAST(gross_calls AS DOUBLE) END) AS y_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN CAST(net_calls   AS DOUBLE) END) AS y_net_calls,
    MAX(CASE WHEN period='yesterday' THEN CAST(orders      AS DOUBLE) END) AS y_orders,

    AVG(CASE WHEN period='prior4' THEN CAST(gross_calls AS DOUBLE) END) AS p4_gross_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(net_calls   AS DOUBLE) END) AS p4_net_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(orders      AS DOUBLE) END) AS p4_orders
  FROM daily_siteserp
  GROUP BY row_name
),

siteserp_out AS (
  SELECT
    d.row_type,
    d.row_name,

    try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)) AS Gross_P4_Mix,
    try_divide(d.y_gross_calls,  NULLIF(t.y_total_gross_calls, 0))  AS Gross_Yday_Mix,
    try_divide(
      try_divide(d.y_gross_calls, NULLIF(t.y_total_gross_calls, 0))
      - try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)),
      NULLIF(try_divide(d.p4_gross_calls, NULLIF(t.p4_total_gross_calls, 0)), 0)
    ) AS Gross_Mix_Pct_Change,

    try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)) AS Net_P4_Mix,
    try_divide(d.y_net_calls,  NULLIF(t.y_total_net_calls, 0))  AS Net_Yday_Mix,
    try_divide(
      try_divide(d.y_net_calls, NULLIF(t.y_total_net_calls, 0))
      - try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)),
      NULLIF(try_divide(d.p4_net_calls, NULLIF(t.p4_total_net_calls, 0)), 0)
    ) AS Net_Mix_Pct_Change,

    try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)) AS Orders_P4_Mix,
    try_divide(d.y_orders,  NULLIF(t.y_total_orders, 0))  AS Orders_Yday_Mix,
    try_divide(
      try_divide(d.y_orders, NULLIF(t.y_total_orders, 0))
      - try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)),
      NULLIF(try_divide(d.p4_orders, NULLIF(t.p4_total_orders, 0)), 0)
    ) AS Orders_Mix_Pct_Change
  FROM period_siteserp_dim d
  CROSS JOIN period_siteserp_totals t
)

SELECT *
FROM category_out
UNION ALL
SELECT *
FROM siteserp_out
ORDER BY
  CASE WHEN row_type='CATEGORY' THEN 1 ELSE 2 END,
  row_name
;
"""

    if debug_print_sql:
        print("--- compute_mix_shifts(): SQL preview ---")
        print(sql[:4000])
        print("--- end SQL preview ---\n")

    return spark.sql(sql)


### Converting SQL Output to Structured JSON

In [0]:
import re
from typing import Any, Dict, List, Optional, Union, Sequence
from pyspark.sql import DataFrame

# -----------------------------
# helpers (unchanged)
# -----------------------------
def _to_snake(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[/%()]", " ", s)
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _safe_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None

def _is_breakdown_kpi(kpi_name: str, delim: str = " - ") -> bool:
    norm = kpi_name.replace("–", "-").replace("—", "-")
    return " - " in norm

def _split_breakdown(kpi_name: str):
    norm = kpi_name.replace("–", "-").replace("—", "-")
    parts = [p.strip() for p in norm.split("-", 1)]
    if len(parts) != 2:
        return (kpi_name.strip(), None)
    return (parts[0].strip(), parts[1].strip())


# -----------------------------
# UPDATED VERSION (adds mixes)
# -----------------------------
def kpi_tables_to_json(
    names: List[str],
    tables: List[Union[str, Any]],  # str table/view name OR Spark DataFrame
    *,
    mixes: Optional[Union[str, DataFrame]] = None,   # <--- NEW
    mixes_key: str = "mix_shifts",                   # <--- NEW
    mixes_max_collect_rows: int = 5000,              # <--- NEW
    kpi_col: str = "KPI",
    current_col: str = "Yesterday",
    baseline_col: str = "P4WA",
    delta_col: str = "Delta",
    call_delta_col: str = "Call_Count_Delta",
    breakdown_reason_key: str = "reason",
    prefer_precomputed_delta: bool = True,
    # Speed / safety knobs
    max_collect_rows: int = 2000,
    use_collect_instead_of_pandas: bool = True,
) -> Dict[str, Any]:
    """
    Builds KPI JSON by section, and (optionally) appends a mix shifts payload at the end.

    mixes table expected columns:
      row_type, row_name,
      Gross_P4_Mix, Gross_Yday_Mix, Gross_Mix_Pct_Change,
      Net_P4_Mix, Net_Yday_Mix, Net_Mix_Pct_Change,
      Orders_P4_Mix, Orders_Yday_Mix, Orders_Mix_Pct_Change
    """
    if len(names) != len(tables):
        raise ValueError(f"names and tables must be same length. Got {len(names)} vs {len(tables)}")

    out: Dict[str, Any] = {}

    # -----------------------------
    # KPI sections (unchanged logic)
    # -----------------------------
    for section_name, tbl in zip(names, tables):
        sdf = spark.table(tbl) if isinstance(tbl, str) else tbl

        cols = set(sdf.columns)
        required = {kpi_col, current_col, baseline_col}
        missing = required - cols
        if missing:
            raise ValueError(f"Section '{section_name}' missing required columns: {sorted(missing)}")

        wanted = [kpi_col, current_col, baseline_col]
        if delta_col in cols:
            wanted.append(delta_col)
        if call_delta_col in cols:
            wanted.append(call_delta_col)

        sdf_small = sdf.select(*wanted)

        n = sdf_small.count()
        if n > max_collect_rows:
            raise ValueError(
                f"Section '{section_name}' has {n:,} rows; refusing to collect to driver. "
                f"Check compute_kpis output (it should be a small KPI table)."
            )

        section_payload = {"metrics": {}, "breakdowns": {}}

        if use_collect_instead_of_pandas:
            rows = sdf_small.collect()
            for r in rows:
                kpi_name = str(r[kpi_col]).strip() if r[kpi_col] is not None else ""
                if not kpi_name:
                    continue

                current_val = _safe_float(r[current_col])
                baseline_val = _safe_float(r[baseline_col])
                pre_delta = _safe_float(r[delta_col]) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None
                calls_delta = _safe_float(r[call_delta_col]) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)
                    section_payload["breakdowns"].setdefault(base_key, [])

                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name,
                    }
                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name,
                    }
                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        else:
            pdf = sdf_small.toPandas()
            for _, row in pdf.iterrows():
                kpi_name = str(row.get(kpi_col, "")).strip()
                if not kpi_name:
                    continue

                current_val = _safe_float(row.get(current_col))
                baseline_val = _safe_float(row.get(baseline_col))
                pre_delta = _safe_float(row.get(delta_col)) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None
                calls_delta = _safe_float(row.get(call_delta_col)) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)
                    section_payload["breakdowns"].setdefault(base_key, [])

                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name,
                    }
                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name,
                    }
                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        out[section_name] = section_payload

    # -----------------------------
    # NEW: append mixes at the end
    # -----------------------------
    if mixes is not None:
        mixes_df = spark.table(mixes) if isinstance(mixes, str) else mixes

        mix_required = {
            "row_type","row_name",
            "Gross_P4_Mix","Gross_Yday_Mix","Gross_Mix_Pct_Change",
            "Net_P4_Mix","Net_Yday_Mix","Net_Mix_Pct_Change",
            "Orders_P4_Mix","Orders_Yday_Mix","Orders_Mix_Pct_Change",
        }
        mix_cols = set(mixes_df.columns)
        missing = mix_required - mix_cols
        if missing:
            raise ValueError(f"mixes missing required columns: {sorted(missing)}")

        mixes_small = mixes_df.select(*[
            "row_type","row_name",
            "Gross_P4_Mix","Gross_Yday_Mix","Gross_Mix_Pct_Change",
            "Net_P4_Mix","Net_Yday_Mix","Net_Mix_Pct_Change",
            "Orders_P4_Mix","Orders_Yday_Mix","Orders_Mix_Pct_Change",
        ])

        m = mixes_small.count()
        if m > mixes_max_collect_rows:
            raise ValueError(
                f"mixes has {m:,} rows; refusing to collect to driver. "
                f"Expected a small output (CATEGORY + SITE_SERP rows)."
            )

        payload = {"CATEGORY": [], "SITE_SERP": [], "OTHER": []}

        rows = mixes_small.collect() if use_collect_instead_of_pandas else mixes_small.toPandas().to_dict("records")
        if use_collect_instead_of_pandas:
            iterable = rows
            def getv(r, k): return r[k]
        else:
            iterable = rows
            def getv(r, k): return r.get(k)

        for r in iterable:
            row_type = (getv(r, "row_type") or "").strip()
            row_name = (getv(r, "row_name") or "").strip()

            item = {
                "row_name": row_name,
                "gross": {
                    "p4_mix": _safe_float(getv(r, "Gross_P4_Mix")),
                    "yday_mix": _safe_float(getv(r, "Gross_Yday_Mix")),
                    "pct_change": _safe_float(getv(r, "Gross_Mix_Pct_Change")),
                },
                "net": {
                    "p4_mix": _safe_float(getv(r, "Net_P4_Mix")),
                    "yday_mix": _safe_float(getv(r, "Net_Yday_Mix")),
                    "pct_change": _safe_float(getv(r, "Net_Mix_Pct_Change")),
                },
                "orders": {
                    "p4_mix": _safe_float(getv(r, "Orders_P4_Mix")),
                    "yday_mix": _safe_float(getv(r, "Orders_Yday_Mix")),
                    "pct_change": _safe_float(getv(r, "Orders_Mix_Pct_Change")),
                },
            }

            if row_type in payload:
                payload[row_type].append(item)
            else:
                payload["OTHER"].append({**item, "row_type": row_type})

        out[mixes_key] = payload

    return out


### JSON Filtering - Removing Unimportant Metrics Before API Call

In [0]:
import math
from typing import Any, Dict, List, Optional, Tuple

def filter_and_compact_kpi_json(
    kpi_json: Dict[str, Any],
    *,
    # -----------------------
    # Filtering thresholds
    # -----------------------
    # Drop whole sections with tiny volume (based on section gross calls)
    min_section_gross_calls: float = 0.0,

    # Metric-level volume gates:
    # Now that per-metric call counts are removed, we use:
    #   - metric["calls_delta"] if present? (not a volume)
    #   - otherwise section gross calls as the volume proxy
    min_metric_current_calls: float = 25.0,

    # Metric impact gates (keeps metric if ANY of these pass)
    min_abs_delta_pct: float = 0.03,          # e.g. 0.03 == 3% relative change
    min_abs_delta_abs: float = 0.0,           # optional for count KPIs; 0 disables if left at 0
    min_abs_delta_pp: float = 0.01,           # for rate-like metrics when delta_abs is “pp” in raw units (0.01==1pp)

    # Always keep these metric keys (snake keys in your JSON), regardless of thresholds
    always_keep_metrics: Tuple[str, ...] = (
        "gross_calls",
        "entered_compass_calls_count",
        "entered_compass_rate",
        "queue_calls",
        "queue_to_gross_rate",
        "abandonment_rate",
        "net_calls",
        "orders",
        "transactional_conversion_rate",
        "gross_conversion_rate",
        "total_revenue",
        "revenue_per_order",
        "revenue_per_net_call",
        "revenue_per_gross_call",
    ),

    # Breakdowns: keep top-N by absolute delta pct points (pp), after filtering
    breakdown_top_n: int = 8,
    min_breakdown_abs_delta_pp: float = 0.01,  # 1pp

    # Since per-breakdown current_calls may also be absent now, this is a best-effort gate:
    # - If item has current_calls, apply threshold
    # - Else, allow it through (or you can choose to gate by section gross)
    min_breakdown_current_calls: float = 25.0,

    # -----------------------
    # Compaction controls
    # -----------------------
    # Removed "current_calls" from defaults; add "calls_delta" instead
    keep_metric_fields: Tuple[str, ...] = ("current", "weekday_baseline_mean", "delta_abs", "delta_pct", "calls_delta"),
    keep_breakdown_fields: Tuple[str, ...] = ("reason", "current", "weekday_baseline_mean", "delta_pct_points", "calls_delta"),
    drop_filter_summary: bool = True,
    drop_empty_breakdowns: bool = True,
    drop_zero_volume_metrics: bool = True,
    round_digits: Optional[int] = 6,
) -> Dict[str, Any]:
    """
    Updated to match KPI JSON after removing per-step call counts:
      - No longer expects metric["current_calls"] / baseline calls.
      - Uses section gross calls as volume proxy for metric filtering.
      - Keeps "calls_delta" (from Call_Count_Delta) when present.

    Assumes input structure like:
      {
        "<section>": {
          "metrics": { "<metric_key>": { ... } },
          "breakdowns": { "<base_key>": [ { ... }, ... ] },
          "filter_summary": { ... }   # optional
        },
        ...
      }
    """

    def _is_nan(x: Any) -> bool:
        return isinstance(x, float) and math.isnan(x)

    def _clean(x: Any) -> Any:
        if x is None:
            return None
        if _is_nan(x):
            return None
        if isinstance(x, float) and math.isinf(x):
            return None
        return x

    def _round(v: Any) -> Any:
        v = _clean(v)
        if v is None:
            return None
        if round_digits is None:
            return v
        if isinstance(v, float):
            return round(v, round_digits)
        return v

    def _get_float(o: Dict[str, Any], key: str) -> Optional[float]:
        v = _clean(o.get(key))
        try:
            return float(v) if v is not None else None
        except Exception:
            return None

    def _metric_volume(metric_obj: Dict[str, Any], section_gross: Optional[float]) -> Optional[float]:
        # Per-metric call counts are gone; use section gross as best proxy.
        return section_gross

    def _metric_keep(metric_key: str, metric_obj: Dict[str, Any], section_gross: Optional[float]) -> bool:
        if metric_key in always_keep_metrics:
            return True

        cur = _get_float(metric_obj, "current")
        if cur is None:
            return False

        if drop_zero_volume_metrics:
            vol0 = _metric_volume(metric_obj, section_gross)
            if vol0 is not None and vol0 == 0:
                return False

        vol = _metric_volume(metric_obj, section_gross)
        if vol is not None and vol < min_metric_current_calls:
            return False

        # Impacts
        d_pct = _get_float(metric_obj, "delta_pct")
        d_abs = _get_float(metric_obj, "delta_abs")  # could be pp for rates or abs change for counts

        passes = False
        if d_pct is not None and abs(d_pct) >= min_abs_delta_pct:
            passes = True
        if (not passes) and (min_abs_delta_abs > 0) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_abs:
            passes = True
        if (not passes) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_pp:
            # For rates, delta_abs is typically pp in raw units (0.01 == 1pp)
            passes = True

        return passes

    def _compact_metric(metric_obj: Dict[str, Any]) -> Dict[str, Any]:
        o = metric_obj or {}
        outm = {}
        for f in keep_metric_fields:
            if f in o:
                outm[f] = _round(o.get(f))
        return {k: v for k, v in outm.items() if v is not None}

    def _compact_breakdown_item(item: Dict[str, Any]) -> Dict[str, Any]:
        it = item or {}
        outb = {}
        for f in keep_breakdown_fields:
            if f in it:
                outb[f] = _round(it.get(f))
        return {k: v for k, v in outb.items() if v is not None}

    filtered_compacted: Dict[str, Any] = {}

    for section_name, section in (kpi_json or {}).items():
        if section_name == 'mix_shifts':
            filtered_compacted['mix_shifts'] = section
        section = section or {}
        metrics_in = section.get("metrics", {}) or {}
        breakdowns_in = section.get("breakdowns", {}) or {}

        # Section gross calls (best-effort)
        section_gross = None
        if "gross_calls" in metrics_in:
            # with new JSON, gross_calls likely has only "current" (and maybe calls_delta)
            section_gross = _get_float(metrics_in["gross_calls"], "current")

        if section_gross is not None and section_gross < min_section_gross_calls:
            continue

        # ---- Metrics: filter + compact ----
        metrics_out: Dict[str, Any] = {}
        for metric_key, metric_obj in metrics_in.items():
            metric_obj = metric_obj or {}

            if not _metric_keep(metric_key, metric_obj, section_gross):
                continue

            cm = _compact_metric(metric_obj)
            if not cm:
                continue

            # Optional final drop: if section volume is 0, drop metric
            if drop_zero_volume_metrics and section_gross is not None and section_gross == 0:
                continue

            metrics_out[metric_key] = cm

        # If a section ends up empty, skip it
        if not metrics_out:
            continue

        # ---- Breakdowns: filter + rank + compact ----
        breakdowns_out: Dict[str, List[Dict[str, Any]]] = {}
        for base_key, items in breakdowns_in.items():
            kept: List[Tuple[float, Dict[str, Any]]] = []

            for item in (items or []):
                item = item or {}
                cur = _get_float(item, "current")
                if cur is None:
                    continue

                # volume gate (best-effort): only apply if current_calls exists
                bvol = _get_float(item, "current_calls")
                if bvol is not None and bvol < min_breakdown_current_calls:
                    continue
                if drop_zero_volume_metrics and bvol is not None and bvol == 0:
                    continue

                dpp = _get_float(item, "delta_pct_points")
                if dpp is None or abs(dpp) < min_breakdown_abs_delta_pp:
                    continue

                kept.append((abs(dpp), item))

            if not kept:
                continue

            kept.sort(key=lambda x: x[0], reverse=True)
            top_items = [it for _, it in kept[:breakdown_top_n]]

            compacted_items = []
            for it in top_items:
                cb = _compact_breakdown_item(it)
                if cb:
                    compacted_items.append(cb)

            if compacted_items:
                breakdowns_out[base_key] = compacted_items

        section_out: Dict[str, Any] = {"metrics": metrics_out}
        if not (drop_empty_breakdowns and not breakdowns_out):
            section_out["breakdowns"] = breakdowns_out

        if not drop_filter_summary and "filter_summary" in section:
            fs = section.get("filter_summary") or {}
            section_out["filter_summary"] = {k: _round(v) for k, v in fs.items() if _round(v) is not None}

        filtered_compacted[section_name] = section_out

    return filtered_compacted


# Report Generation From JSON

## First Pass

### Previous Reports

Just looping through the previous 7 days reports and summarizing them - almost executing level, but making sure to note small trends as well to spot continuity. 

In [0]:
from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from docx import Document


_SUMMARY_RE = re.compile(r"^(?P<mm>\d{2})_(?P<dd>\d{2})_(?P<yyyy>\d{4})_summary\.docx$", re.IGNORECASE)


def _parse_summary_date_from_filename(filename: str) -> Optional[datetime]:
    m = _SUMMARY_RE.match(filename)
    if not m:
        return None
    mm, dd, yyyy = int(m["mm"]), int(m["dd"]), int(m["yyyy"])
    return datetime(yyyy, mm, dd)


def _read_docx_text(docx_path: Path) -> str:
    doc = Document(str(docx_path))
    parts: List[str] = []
    for p in doc.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    return "\n".join(parts)


def _extract_json_from_doc_text(doc_text: str) -> Dict[str, Any]:
    """
    Best-effort extraction of a JSON object from a docx text blob.
    Assumes the doc contains a JSON block OR at least one {...} object.
    """
    doc_text = doc_text.strip()

    # If the whole thing is JSON, great.
    try:
        obj = json.loads(doc_text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Otherwise, try to extract the first JSON object substring.
    start = doc_text.find("{")
    end = doc_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in summary docx text.")
    candidate = doc_text[start : end + 1]
    obj = json.loads(candidate)
    if not isinstance(obj, dict):
        raise ValueError("Extracted JSON is not an object.")
    return obj
def get_weekly_context(
    knowledge_dir: str | Path = "knowledge",
    lookback_summaries: int = 7,
) -> Dict[str, Any]:
    """
    Reads the most recent N summary .docx files in knowledge/summaries named:
      MM_DD_YYYY_summary.docx

    Path resolution rules (notebook/DBX-safe):
    - If `knowledge_dir` is absolute, use it as-is.
    - If `knowledge_dir` is relative (default "knowledge"), try:
        A) Walk up from CWD looking for a directory that contains knowledge/summaries.
        B) Prefer the parent of that candidate (one level up) when it also has knowledge/summaries.
       This ensures we pick the repo root that lives *outside* src when the code runs under src.

    Returns:
      {"weekly_context": {"MM_DD_YYYY": {"executive_summary": "<full doc text>"}, ...}}

    If duplicate date keys occur, the newest file (by filename date) wins.
    """
    knowledge_dir = Path(knowledge_dir)

    def _find_candidate_with_knowledge(start: Path) -> Optional[Path]:
        """
        Walk up from start and return the first parent that contains knowledge/summaries.
        If none found, return None.
        """
        cur = start.resolve()
        for parent in [cur] + list(cur.parents):
            if (parent / "knowledge" / "summaries").exists():
                return parent
        return None

    if not knowledge_dir.is_absolute():
        cwd = Path.cwd()
        candidate = _find_candidate_with_knowledge(cwd)

        # If we found a candidate, prefer candidate.parent if it contains the folder (one level up).
        if candidate:
            parent_candidate = candidate.parent
            if (parent_candidate / "knowledge" / "summaries").exists():
                repo_root = parent_candidate
            else:
                repo_root = candidate
        else:
            # No candidate found walking up. Fallback: if cwd contains a "src" component,
            # try to use the directory above "src" (common layout).
            parts = cwd.resolve().parts
            if "src" in parts:
                idx = parts.index("src")
                if idx > 0:
                    repo_root = Path(*parts[:idx])
                else:
                    repo_root = cwd.resolve().parent
            else:
                # As a last resort, just use the cwd parent (one level up from working dir)
                repo_root = cwd.resolve().parent

        knowledge_dir = repo_root / knowledge_dir

    summaries_dir = knowledge_dir / "summaries"
    summaries_dir.mkdir(parents=True, exist_ok=True)

    # Gather dated summary files
    dated: List[Tuple[datetime, Path]] = []
    for p in summaries_dir.glob("*_summary.docx"):
        dt = _parse_summary_date_from_filename(p.name)
        if dt:
            dated.append((dt, p))

    if not dated:
        return {"weekly_context": {}}

    # Sort newest -> oldest; take top N
    dated.sort(key=lambda x: x[0], reverse=True)
    selected = dated[:lookback_summaries]

    merged: Dict[str, Any] = {}
    for dt, path in selected:
        # Read the full docx text
        try:
            doc_text = _read_docx_text(path)
        except Exception as e:
            # Skip files we can't open, but continue processing others
            # (you can change this to raise if you prefer strict behavior)
            print(f"Warning: failed to read {path}: {e}")
            continue

        # Key derived from the filename date in MM_DD_YYYY form
        # Use the filename's mm_dd_yyyy as the date_key to match previous shape
        filename = path.name
        mm = f"{dt.month:02d}"
        dd = f"{dt.day:02d}"
        yyyy = f"{dt.year:04d}"
        date_key = f"{mm}_{dd}_{yyyy}"

        # Build the payload as plain text (no JSON expected)
        payload = {"executive_summary": doc_text}

        # Newer summaries override older ones for the same date key.
        merged[date_key] = payload

    print("Found and parsed", len(merged), "weekly summaries")
    print(merged)

    return {"weekly_context": merged}


### Getting Other Business Context

Basically will make a small OpenAI call just to get proper Vectore Store queries. The returned blocks here will be passed along to the actual query for context. 

This call has a small context block hard-coded so that the data can at least be interpreted.

Getting vector store queries from OpenAI call.

In [0]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI


def _find_dotenv(start_path: Path) -> Optional[Path]:
    """Walk upward from start_path looking for a .env file."""
    for parent in [start_path] + list(start_path.parents):
        candidate = parent / ".env"
        if candidate.exists():
            return candidate
    return None


def _load_env(dotenv_path: str | Path | None = None) -> None:
    """Load environment variables from .env (explicitly)."""
    if dotenv_path is not None:
        p = Path(dotenv_path)
        if not p.exists():
            raise FileNotFoundError(f".env not found at: {p}")
        load_dotenv(p, override=True)
        return

    found = _find_dotenv(Path.cwd())
    if found:
        load_dotenv(found, override=True)


def generate_vector_store_queries(
    *,
    kpi_json: Dict[str, Any],
    weekly_context: Optional[Dict[str, Any]] = None,
    base_business_context: str,
    n_queries: int = 10,
    model: str = "gpt-5-mini",
    # NEW:
    dotenv_path: str | Path | None = None,
    vector_store_env_var: str = "OPENAI_VECTOR_STORE_ID",
) -> Dict[str, Any]:
    """
    Uses OpenAI to turn (1) daily KPI JSON + (2) weekly summaries + (3) minimal business context
    into a set of vector-store search queries for retrieving more context.

    Loads OPENAI_API_KEY from .env via load_dotenv (or uses existing env if already set).
    Also reads OPENAI_VECTOR_STORE_ID from env and returns it for downstream retrieval functions.

    Returns:
      {
        "vector_store_id": "vs_..." | null,
        "queries": [
          {"query": "...", "why": "...", "priority": "High|Medium|Low", "filters": {...}},
          ...
        ]
      }
    """
    # Ensure env vars are present (OPENAI_API_KEY / OPENAI_VECTOR_STORE_ID)
    _load_env(dotenv_path)

    client = OpenAI()  # will read OPENAI_API_KEY from env

    system_instructions = (
        "You are an analyst assistant that generates targeted semantic-search queries for a vector store. "
        "Your job is to propose queries that will retrieve the missing business context needed to explain "
        "today's performance in a daily Energy Voice report."
    )

    user_prompt = f"""
Use the following inputs to generate exactly {n_queries} vector-store search queries.

INPUTS
1) Minimal business context (shared vocabulary + directional interpretation):
{base_business_context}

2) Today's KPI JSON (the data we are explaining):
{json.dumps(kpi_json, ensure_ascii=False)}

3) Weekly context (prior 7 daily summaries; may be empty):
{json.dumps(weekly_context or {}, ensure_ascii=False)}

TASK
Generate EXACTLY {n_queries} search queries to retrieve additional context from our knowledge base.
The queries should focus on explaining the likely drivers of today's KPI changes and any notable trends.
Use Energy Voice vocabulary (Compass, Q2G, ERCOT match, abandon, CIContact, CICredit, RPO, RPGC, partner mix, routing).

QUERY GUIDELINES
- Each query must be short (4–12 words) and specific.
- Avoid numbers and dates unless essential.
- Prefer “drivers/causes/playbook/experiment/routing/partner mix/incidents” style phrasing.
- Diversity: cover funnel stages (marketing mix → Compass/IVR → queue → agent → revenue).
- If you suspect an explanation could be an incident or program change, include at least one query for that.

OPTIONAL FILTERS (include only if helpful)
- doc_type: one of ["glossary","playbook","experiment","incident","partner_update","ops_update","faq"]
- product_area: one of ["marketing","compass","queue","agent","revenue","routing","partner_mix"]
- recency_days: integer (e.g., 30, 60, 90) when freshness matters

OUTPUT
Return STRICT JSON with this shape:
{{
  "queries": [
    {{
      "query": "...",
      "why": "1 sentence rationale",
      "priority": "High|Medium|Low",
      "filters": {{"doc_type": "...", "product_area": "...", "recency_days": 60}}
    }}
  ]
}}
""".strip()

    schema = {
        "name": "vector_store_query_plan",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "queries": {
                    "type": "array",
                    "minItems": n_queries,
                    "maxItems": n_queries,
                    "items": {
                        "type": "object",
                        "properties": {
                            "query": {"type": "string"},
                            "why": {"type": "string"},
                            "priority": {"type": "string", "enum": ["High", "Medium", "Low"]},
                            "filters": {
                                "anyOf": [
                                    {
                                    "type": "object",
                                    "properties": {},
                                    "additionalProperties": False
                                    },
                                    {
                                    "type": "object",
                                    "properties": {
                                        "doc_type": {
                                        "type": ["string", "null"],
                                        "enum": [
                                            "glossary","playbook","experiment","incident",
                                            "partner_update","ops_update","faq", None
                                        ]
                                        },
                                        "product_area": {
                                        "type": ["string", "null"],
                                        "enum": [
                                            "marketing","compass","queue","agent",
                                            "revenue","routing","partner_mix", None
                                        ]
                                        },
                                        "recency_days": {"type": ["integer", "null"], "minimum": 1, "maximum": 3650}
                                    },
                                    "required": ["doc_type", "product_area", "recency_days"],
                                    "additionalProperties": False
                                    }
                                ]
                                }
                                ,
                        },
                        "required": ["query", "why", "priority", "filters"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["queries"],
            "additionalProperties": False,
        },
    }

    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": schema["name"],
                "strict": True,
                "schema": schema["schema"],
            }
        },
    )

    raw = (resp.output_text or "").strip()
    try:
        plan = json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Model did not return JSON. Output was:\n{raw}")
        plan = json.loads(raw[start : end + 1])

    return {
        "vector_store_id": os.environ.get(vector_store_env_var),
        "queries": plan["queries"],
    }


Querying the vector store

In [0]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI

# reuse your existing _find_dotenv / _load_env if defined; otherwise simple loader:
def _find_dotenv(start_path: Path) -> Optional[Path]:
    for parent in [start_path] + list(start_path.parents):
        candidate = parent / ".env"
        if candidate.exists():
            return candidate
    return None

def _load_env(dotenv_path: str | Path | None = None) -> None:
    if dotenv_path is not None:
        p = Path(dotenv_path)
        if not p.exists():
            raise FileNotFoundError(f".env not found at: {p}")
        load_dotenv(p, override=True)
        return
    found = _find_dotenv(Path.cwd())
    if found:
        load_dotenv(found, override=True)

def _safe_get(obj: Any, attr: str, default=None):
    """Try dict access then attribute access."""
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)

def _item_to_dict(item: Any) -> Dict[str, Any]:
    """Convert SDK item to plain dict for introspection and access."""
    if item is None:
        return {}
    if isinstance(item, dict):
        return item
    # pydantic v2
    if hasattr(item, "model_dump"):
        try:
            return item.model_dump()
        except Exception:
            pass
    # pydantic v1
    if hasattr(item, "dict"):
        try:
            return item.dict()
        except Exception:
            pass
    # fallback
    if hasattr(item, "__dict__"):
        return {k: v for k, v in item.__dict__.items() if not k.startswith("_")}
    return {"repr": repr(item)}

def _extract_text_from_content(content: List[Any]) -> str:
    """
    Tolerant extractor: accepts dicts or objects, tries common field names and types.
    """
    parts: List[str] = []
    if not content:
        return ""
    for c in content:
        # Normalize to dict-ish view for easier checks
        cdict = _item_to_dict(c)
        # Common explicit 'text' segments
        ctype = cdict.get("type") or cdict.get("segment_type") or cdict.get("kind") or None
        ctext = cdict.get("text") or cdict.get("content") or cdict.get("body") or cdict.get("value") or None

        # If type is explicitly a text-like type, accept it
        if ctype and isinstance(ctype, str) and ctype.lower() in ("text", "plain_text", "paragraph", "content"):
            if isinstance(ctext, str) and ctext.strip():
                parts.append(ctext.strip())
                continue

        # If there was no explicit type, but there is a text-like field, accept it
        if isinstance(ctext, str) and ctext.strip():
            parts.append(ctext.strip())
            continue

        # Some SDK shapes present nested structures; attempt to stringify fallback
        # e.g., {"data": [{"text":"..."}]} or object fields
        # Try to find deepest string values heuristically
        if isinstance(cdict, dict):
            for candidate_key in ("text", "content", "body", "value", "excerpt"):
                cand = cdict.get(candidate_key)
                if isinstance(cand, str) and cand.strip():
                    parts.append(cand.strip())
                    break
            else:
                # Last resort: flatten small primitives in the dict
                flat_vals = []
                for v in cdict.values():
                    if isinstance(v, str) and v.strip():
                        flat_vals.append(v.strip())
                    elif isinstance(v, (int, float)):
                        flat_vals.append(str(v))
                if flat_vals:
                    parts.append(" ".join(flat_vals))
    return "\n".join(parts).strip()

def _build_vector_store_filters(file_attrs: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Same behavior as your previous builder; skip recency_days."""
    if not file_attrs:
        return None
    keep = {k: v for k, v in file_attrs.items() if v not in (None, "", [], {})}
    if not keep:
        return None
    comparisons: List[Dict[str, Any]] = []
    for k, v in keep.items():
        if k == "recency_days":
            continue
        comparisons.append({"type": "eq", "key": k, "value": v})
    if not comparisons:
        return None
    if len(comparisons) == 1:
        return comparisons[0]
    return {"type": "and", "filters": comparisons}

def retrieve_vector_store_blocks(
    *,
    vector_store_id: str | None = None,
    query_plan: Dict[str, Any],
    max_results_per_query: int = 5,
    score_threshold: float = 0.0,
    dedupe_on_text: bool = True,
    save_path: str | Path | None = None,
    dotenv_path: str | Path | None = None,
    vector_store_env_var: str = "OPENAI_VECTOR_STORE_ID",
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Robust retrieval: tries both attribute_filter and filters params and tolerates varied segment shapes.
    """
    _load_env(dotenv_path)

    if not vector_store_id:
        vector_store_id = os.environ.get(vector_store_env_var)
    if not vector_store_id:
        raise ValueError(
            "vector_store_id was not provided and no vector store id was found in env. "
            f"Set {vector_store_env_var}=vs_... in your .env or pass vector_store_id=..."
        )

    client = OpenAI()

    queries = query_plan.get("queries", [])
    if not isinstance(queries, list) or not queries:
        raise ValueError("query_plan must contain a non-empty 'queries' list.")

    blocks: Dict[str, Any] = {}
    seen_texts: set[str] = set()
    block_num = 1

    # Helper that tries several search parameter names and returns the result tuple (res, used_param)
    def _search_with_tryouts(qtext: str, attr_filter: Optional[Dict[str, Any]]):
        # 1) Try attribute_filter (newer SDK)
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                ranking_options={"score_threshold": score_threshold},
            )
            # If it returned without throwing, return it (we'll check data length)
            return res, "attribute_filter"
        except Exception as e:
            if verbose:
                print(f"[search] attribute_filter raised: {type(e).__name__}: {e}")
        # 2) Try filters (older SDK)
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                filters=attr_filter,
                ranking_options={"score_threshold": score_threshold},
            )
            return res, "filters"
        except Exception as e:
            if verbose:
                print(f"[search] filters raised: {type(e).__name__}: {e}")
        # 3) Last resort: try no filters at all
        try:
            res = client.vector_stores.search(
                vector_store_id=vector_store_id,
                query=qtext,
                max_num_results=max_results_per_query,
                ranking_options={"score_threshold": score_threshold},
            )
            return res, "no_filters"
        except Exception as e:
            # Give up for this query
            if verbose:
                print(f"[search] no-filters raised: {type(e).__name__}: {e}")
            return None, None

    for qi, q in enumerate(queries, start=1):
        query_text = (q.get("query") or "").strip()
        if not query_text:
            if verbose:
                print(f"[{qi}] skipping empty query object keys={list(q.keys())}")
            continue

        raw_filters = q.get("filters") or {}
        if not isinstance(raw_filters, dict):
            raw_filters = {}
        attribute_filter = _build_vector_store_filters(raw_filters)

        if verbose:
            print(f"[{qi}] searching for: {query_text!r} (filters={raw_filters})")

        res, used = _search_with_tryouts(query_text, attribute_filter)
        if res is None:
            if verbose:
                print(f"[{qi}] search attempts failed for query: {query_text!r}")
            continue

        # Normalize results to list
        res_data = getattr(res, "data", None) or _item_to_dict(res).get("data") or []
        nres = len(res_data or [])
        if verbose:
            print(f"[{qi}] search returned {nres} result(s) using param: {used}")

        if nres == 0:
            # Nothing returned for this query — continue to next query
            if verbose:
                print(f"[{qi}] no hits for query_text={query_text!r}")
            continue

        # Process each returned item
        for ri, item in enumerate(res_data, start=1):
            item_d = _item_to_dict(item)
            content = item_d.get("content") or []
            # Fallback: some shapes put text at item_d.get("text") directly
            if not content and isinstance(item_d.get("text"), str):
                content = [{"type": "text", "text": item_d.get("text")}]

            chunk_text = _extract_text_from_content(content)
            if not chunk_text:
                # If no chunk text, try to look for nested content keys
                # e.g., item_d.get("contents") or item_d.get("segments")
                for alt in ("contents", "segments", "data"):
                    altv = item_d.get(alt)
                    if altv:
                        chunk_text = _extract_text_from_content(altv)
                        if chunk_text:
                            break

            if not chunk_text:
                # nothing to add from this item
                if verbose:
                    print(f"[{qi}.{ri}] item had no extractable text; keys: {list(item_d.keys())}")
                continue

            if dedupe_on_text:
                norm = " ".join(chunk_text.split())
                if norm in seen_texts:
                    if verbose:
                        print(f"[{qi}.{ri}] duplicate text skipped")
                    continue
                seen_texts.add(norm)

            score = item_d.get("score")
            file_id = item_d.get("file_id")
            filename = item_d.get("filename")
            attributes = item_d.get("attributes") or {}

            blocks[str(block_num)] = {
                "block_id": block_num,
                "text": chunk_text,
                "score": score,
                "file_id": file_id,
                "filename": filename,
                "attributes": attributes or {},
                "retrieval": {
                    "query_index": qi,
                    "query": query_text,
                    "result_rank": ri,
                    "priority": q.get("priority"),
                    "why": q.get("why"),
                    "requested_filters": raw_filters,
                    "applied_filters": attribute_filter,
                    "search_param_used": used,
                },
            }
            if verbose:
                print(f"[{qi}.{ri}] added block #{block_num} (score={score}) filename={filename}")
            block_num += 1

    artifact = {"blocks": blocks}

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")

    return artifact


## Second Pass

Putting it all together here - this is where prompting gets pretty tough. Gonna start with executive summary, then do drivers, then potential analyses, then potentially figures/pacing numbers. 

### Aggregating Context

In [0]:
from __future__ import annotations
from datetime import datetime, timedelta, time as dtime
from typing import Any, Dict, Optional
from slack_sdk import WebClient
from zoneinfo import ZoneInfo


BUSINESS_CONTEXT_OVERVIEW = """
Part I — Business Context Overview
(Marketplace, Channels, Compass, and Partners)
1. Who We Are as a Business
Energy Voice operates an online energy marketplace for deregulated Texas electricity.
We are not a utility and not a Retail Electric Provider (REP). We are a marketplace and distributor connecting customers with providers.
Value Proposition to Customers
Choice: Compare plans across multiple REPs
Guidance: Help customers understand plans, pricing, and eligibility
Convenience: Enroll digitally or by phone without dealing with each provider individually
Value Proposition to Providers
Customer acquisition at scale
Qualified demand (address-validated, credit-screened, intent-driven)
Performance-based economics (we are paid when orders convert and remain active)

2. Two Sides of the Marketplace: Site + Phone
Energy Voice operates across two tightly connected channels:
Digital (Site)
Brands include SaveOnEnergy (SOE), CompareTexasPower (CTXP), and others
Captures demand via SEO, paid search, affiliates, and partners
Users compare plans and sometimes complete enrollment online
Generates high-quality intent signals (zip, address, product interest)
Voice (Phone)
Converts users who:
Prefer human assistance
Have higher complexity (moving, credit issues)
Drop out of digital flows
Higher cost than digital, but higher revenue per interaction
Critical for scale, yield, and partner commitments
Key principle: Site and phone are not separate businesses. Phone is an extension and rescue layer of digital.

3. How a Voice Call Flows (System-Level)
A call passes through three platforms before it becomes revenue:
1️⃣ Twilio — Telephony Layer
Handles inbound calls and routing
First interaction includes:
Language selection (English / Spanish)
Call metadata (number, source, timestamps)
Routes caller into Compass IVR
Twilio answers the phone, but does not decide outcomes. Only one question is in here, so it is not very important. Compass is the real IVR experience.

2️⃣ Compass (Voiceflow) — First Touch / Intelligence Layer
Compass is our AI-powered IVR and intelligence layer.
Responsibilities:
Identify serviceability (Texas)
Collect and confirm customer info
Enrich calls with site and historical context
Shape intent before the agent
Weed out bot calls (Done at first question - Initial Question)
Compass determines whether a call is:
Ready for a sales agent
Worth agent time
Compass is fundamentally different from legacy IVRs because it is intent-aware, context-aware, and continuously iterated.

3️⃣ Agent — Sales Layer
Agents receive:
A pre-qualified, enriched call
Context from Compass (address, name, intent, hints)
Agents complete:
Contact info, Call Into Contact
Credit check, Call Into Credit
Product pitch
Enrollment, Sale
Compass does not replace agents — it makes agents more productive and selective.

4. Sale → Revenue
Order placed with a provider
Revenue generated based on partner economics
Long-term value depends on:
Product type
Credit quality
Customer retention

5. Why Compass Exists
Before Compass
Shallow, rule-based IVRs
Agents wasted time on:
Unserviceable calls
Low-intent calls
Re-collecting information
High abandon rates and poor capacity utilization
With Compass
Compass is designed to:
Increase serviceability
Reduce agent waste
Improve routing decisions
Increase Revenue per Gross Call (RPGC)
Compass is still relatively new compared to the rest of the voice stack, which is why:
We run frequent split tests
We monitor funnel shifts closely
We are cautious about unintended downstream impacts
Compass is a living system, not a static tool.

6. Provider Partnerships (How We Make Money)
Vistra — Primary Strategic Partner
Key Vistra brands:
TXU Energy
Tri-Eagle Energy (TEE)
Why Vistra matters:
Strong brand recognition
Broad product availability
Competitive economics
High Revenue per Order (RPO)
As a result:
We intentionally lead with Vistra products
Agent training and scripting are optimized for Vistra
Compass and routing strategies often prioritize Vistra-eligible calls
Other Providers
Improve coverage and optionality
Help convert failed-credit callers
Typically lower RPO and/or higher churn risk

7. How the Business Defines Success
At a high level:
Marketing determines who calls
Compass determines who deserves an agent and builds intent
Agents determine who converts
Partners and products determine how much we earn
Everything ladders up to:
Revenue per Gross Call (RPGC)
Scalable growth without exploding costs

Non-Brand Buckets: These are customers who are searching for electricity they tend to have a higher conversion rate because they are actively shopping for new service or wanting to switch service and they don’t have a specific brand or supplier in mind. They also are made up mostly of callers who have visited our site before calling in. We tend to see this volume make up around 50-55% of total volume, when the mix of this volume increases we tend to see performance go up because our average intent of calls goes up in correlation.

The main segments of the non-brand buckets are:
Aggregator: which is typically our highest intent and highest converting audience and converts around 47%-50% net conversion
Generic: which is also high converting around 34%-36% net conversion
Natural: which are customers who are coming through our site not from a paid search campaign and they convert at around the same rate as the Generic audience.

Brand Buckets: These are customers that searched a particular retail electric provider in the search engine results page (SERP) of google or bing and are either existing customers of the retail electric provider that they searched or are customers interested in that particular brand. We tend to see the sum of this volume make up around 45-50% of queue call volume. When mix of this volume rises, that causes declines in net conversion due to our average intent of callers going down.

These customers convert lower then non brand customers because they have less intent, they usually convert around 30%. The other reason that they convert lower is because most of these customers don’t visit our site and fall into our SERP bucket which means they called directly from the search engine results page without ever visiting our website to see the offers we have. We tend to see our SERP bucket mix be around 45-50% of volume and correlates closely with Brand volume mix. Our SERP bucket converts around 25% whereas our Site bucket converts around 45%.

The main segments of the brand audience are:
Brand Partner - which are customers who searched a retail electric provider that we offer as an option to purchase over the phone or on our website. This audience typically converts at around 30%.
Competitor: these are customers who searched a retail electric provider that we do not offer as an option to purchase over the phone or on our website. This audience typically converts at 25-27%.
Utility: These are customers who searched one of the Utility providers in Texas and typically are very low converting because they are either unserviceable or calling about an outage in their area and not about purchasing a new energy plan. They typical convert around 20%
""".strip()


BUSINESS_CONTEXT_GLOSSARY = """
Part II — Energy Voice Glossary
(Marketing → Compass → Agent → Revenue)
1. Marketing Buckets (Where Calls Come From)
Brand
Searches for a specific provider (e.g., “TXU Energy phone number”).
Strength: High intent
Risk: Provider confusion
Non-Brand (NB)
Generic category searches (e.g., “cheap electricity Texas”).
Strength: Scales well
Risk: Price sensitivity
Aggregator
Third-party comparison partners.
Strength: Historically strong conversion
Risk: Volume volatility
Competitor
Searches referencing competitors.
Strength: Switching intent
Risk: Skepticism
Natural / Organic
SEO-driven unpaid traffic.
Strength: High margin
Risk: Mixed intent
PMax / Display / Other
Broad paid media.
Strength: Volume expansion
Risk: Lower intent

2. Site vs SERP (Call Origin Context)
SERP
Caller clicks “Call” directly from Google.
No site data
Higher reliance on IVR
Site
Caller visited SOE / CTXP before calling.
Rich enrichment passed into Compass
Stronger downstream performance

3. Compass / IVR Metrics
Gross Call
Any inbound call hitting the phone system.
IVR Flow (Simplified)
Twilio → Language → Home/Business → Texas Check → Moving/Switching → Zip → Address → Name → DoB → Queue
Q2G (Queue-to-Gross)
% of gross calls reaching queue.
Declines signal IVR friction
ERCOT Match
Address successfully matches a valid ERCOT service location.
Address / Name Collection
Successful capture and confirmation of caller details.
Customer Enrichment
Prefill or confirmation using site or historical data.

4. Queue & Staffing Metrics
Queue Call
Call waiting for an agent.
Abandon Rate (AR)
% of queued calls abandoned.
Healthy: ~2–3%
Net Call
Call that reaches an agent.
Net Calls = Gross − IVR Drop − Abandons

5. Agent Funnel Metrics
CIContact
Agent completes contact info.
CICredit
Caller agrees to credit check.
Passed Credit Conversion
% of credit runs that pass.
Failed Credit Conversion
% of failed-credit callers that still convert.
Post-Credit Conversion
% converting after passing credit.
Net Conversion (NC)
Orders ÷ Net Calls
Gross Conversion (GC)
Orders ÷ Gross Calls

6. Revenue & Yield Metrics
Order
Completed enrollment.
Revenue per Order (RPO)
Average revenue per order.
Revenue per Gross Call (RPGC)
Revenue ÷ Gross Calls
North Star metric
Contribution Margin (CM)
Revenue minus variable costs.

7. Agent & Ops Concepts
Agent Tenure
New / Nesting: <30–60 days
Veteran: Higher consistency
Routing & Prioritization
Controls which calls go to which agents and in what order.

8. Holistic Mental Model
Marketing → Compass → Agent → Partner Economics → Revenue
All analysis should reason down-funnel and up-funnel simultaneously, with RPGC as the ultimate scorecard.
""".strip()


ANALYST_DATA_CONTEXT = """
===== ANALYST DATA CONTEXT (Energy Voice) =====

DATA PRESENCE & SEMANTICS
- If a metric or field is missing from the input JSON, it was intentionally removed
  because it was insignificant for the period being analyzed.
- Missing fields should NOT be called out as errors or omissions unless their absence
  creates real analytical ambiguity.

KNOWN SEASONAL / STRUCTURAL EFFECTS
- Call volume often lifts at the beginning of the month due to:
  - Customers calling about existing plans
  - Customers setting up new service
- Treat this as light seasonal context; do not over-attribute daily performance to it.

BOTS & TOP-OF-FUNNEL NOISE
- Automated or bot traffic is filtered out at the first Compass question.
- Bots never reach the queue and never impact agent metrics.
- Bot volume should be treated as top-of-funnel noise only.
- Compass doesn't 'allow' or 'disallow' any calls - if bot calls are up or down, it is a completely external factor.
- Bot volume should not be treated as a queue or net metric, but filtering bots can give queue rates a more stable view.

CANONICAL FUNNEL DEFINITIONS
- A “contact call” is a net call where all required contact information is completed.
- A “credit call” is a net call where credit has been successfully run.
- The net call funnel always progresses in this order:
  contact → credit → pitch → conversion
- Passed credit rate is the rate at which customers pass a credit check. If they fail, the types of products we can sell them shrinks drastically and gives us less revenue per sale.
- Passed credit conversion is conversion is the rate at which credit passed customers convert. 
- Failed credit conversion is conversion is the rate at which credit failed customers convert. 

COMPASS INTERPRETATION PRINCIPLES
- The Compass initial question is the most important and most nuanced step in the IVR.
- This step naturally has the highest breakage and is also where bots are filtered.
- Higher breakage at the initial question is not inherently negative and must be
  interpreted in the context of call quality and downstream outcomes.

ANALYTICAL BIAS TO AVOID, TIPS FOR INTERPRETATION
- Do not assume funnel drop-off equals failure.
- Always evaluate Compass behavior in terms of how it shapes call quality,
  agent exposure, and revenue outcomes.
- Do not only look at intr-bucket declines to explain performance. Mix shifts to lower converting buckets will impact performance more often that true performance drop-offs. 
- Do not just state 'net conversion is down' - which step in the net funnel is it down in? Which buckets? Which steps are up? What may be driving that?

===== FORMATTING INSTRUCTIONS =====
- All numbers in the thousands should be formatted as "1,234", with a comma. Numbers this large should not include any decimal places. 
- All decimal values should be formatted as decimals, rounded to 1 decimal place (ex: 0.1234 -> 12.3%)
- These formatting instructions should be applied to all sections of the report, even the evidence section. 
- All section headers should have the first letters capitalized as in a normal title (ex: Month-to-Date Pacing, Executive Summary, etc.)
""".strip()


def _format_vector_blocks(
    retrieved_blocks: Optional[Dict[str, Any]],
    *,
    max_blocks: int = 8,
    max_chars_per_block: int = 2000,
) -> str:
    """
    Normalizes your vector DB retrieval payload into stable, readable prompt text.

    Expected shape (loose):
      retrieved_blocks = {
        "blocks": {
          "<any_key>": {"block_id": "1", "text": "...", ...},
          ...
        },
        ...
      }
    """
    if not retrieved_blocks:
        return ""

    blocks = retrieved_blocks.get("blocks", {})
    if not isinstance(blocks, dict) or not blocks:
        return ""

    # Prefer deterministic ordering by block_id if present.
    try:
        ordered = sorted(blocks.values(), key=lambda b: int((b or {}).get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())

    out = []
    for i, b in enumerate(ordered[:max_blocks]):
        if not isinstance(b, dict):
            continue
        text = (b.get("text") or "").strip()
        if not text:
            continue
        if len(text) > max_chars_per_block:
            text = text[:max_chars_per_block] + " ..."
        out.append(f"BLOCK[{i+1}]: {text}")

    return "\n\n".join(out)

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta, time as dtime
from typing import Any, Dict, List, Optional, Tuple

from zoneinfo import ZoneInfo


@dataclass(frozen=True)
class SlackMessage:
    ts: str
    user_id: Optional[str]
    username: Optional[str]
    text: str
    subtype: Optional[str] = None
    bot_id: Optional[str] = None


def _slack_window_for_daily_job(
    *,
    run_dt: datetime,
    tz: ZoneInfo,
    start_time_local: dtime = dtime(9, 0),
) -> Tuple[float, float, datetime, datetime]:
    """
    Daily job window:
      start = yesterday @ 9:00 AM ET
      end   = run_dt (job runs ~7:00 AM ET, but you may pass actual now)
    """
    if run_dt.tzinfo is None:
        run_dt = run_dt.replace(tzinfo=tz)
    yesterday = (run_dt - timedelta(days=1)).date()
    start_local = datetime.combine(yesterday, start_time_local).replace(tzinfo=tz)
    return start_local.timestamp(), run_dt.timestamp(), start_local, run_dt


def _extract_best_effort_text(m: Dict[str, Any]) -> str:
    """
    Slack messages can be "empty text" but still meaningful (blocks/attachments/files).
    This tries to create a useful short representation.
    """
    text = (m.get("text") or "").strip()
    if text:
        return text

    # Attachments
    attachments = m.get("attachments") or []
    if attachments:
        parts: List[str] = []
        for a in attachments[:3]:
            if not isinstance(a, dict):
                continue
            t = (a.get("title") or "").strip()
            pre = (a.get("pretext") or "").strip()
            fallback = (a.get("fallback") or "").strip()
            if t:
                parts.append(t)
            elif pre:
                parts.append(pre)
            elif fallback:
                parts.append(fallback)
        if parts:
            return " | ".join(parts)

    # Files
    files = m.get("files") or []
    if files:
        names: List[str] = []
        for f in files[:3]:
            if not isinstance(f, dict):
                continue
            n = (f.get("name") or f.get("title") or f.get("mimetype") or "").strip()
            if n:
                names.append(n)
        if names:
            return f"(files shared: {', '.join(names)})"
        return "(files shared)"

    # Blocks (very rough)
    blocks = m.get("blocks") or []
    if blocks:
        # Try to pull first few plaintext snippets
        snippets: List[str] = []
        for b in blocks[:5]:
            if not isinstance(b, dict):
                continue
            # Common structure: block -> elements -> text
            txt = ""
            if "text" in b and isinstance(b["text"], dict):
                txt = (b["text"].get("text") or "").strip()
            if not txt:
                elements = b.get("elements") or []
                for el in elements:
                    if isinstance(el, dict) and el.get("type") == "text":
                        txt = (el.get("text") or "").strip()
                        if txt:
                            break
            if txt:
                snippets.append(txt)
        if snippets:
            return " ".join(snippets)[:500].strip() or "(message contains blocks)"
        return "(message contains blocks)"

    return ""

from typing import Iterable, Union, Set

def _fetch_slack_messages_last_window(
    *,
    channel_id: str,
    slack_api_key: Optional[str] = None,
    run_dt: Optional[datetime] = None,
    max_messages: int = 400,
    debug: bool = False,
    exclude_bot_ids: Optional[Union[str, Iterable[str]]] = "B0ABK6BCWRY",  # accept string or iterable
) -> List[SlackMessage]:
    """
    Fetcher that excludes any messages whose bot_id is in exclude_bot_ids.

    exclude_bot_ids may be:
      - None (don't exclude any)
      - a single bot_id string
      - an iterable of bot_id strings (list/set/tuple)
    """
    import os
    from dotenv import load_dotenv
    from slack_sdk import WebClient
    from slack_sdk.errors import SlackApiError

    load_dotenv()
    token = slack_api_key or os.getenv("SLACK_API_KEY")
    if not token:
        if debug:
            print("[slack] Missing SLACK_API_KEY.")
        return []

    # normalize exclude_bot_ids -> set or empty set
    if exclude_bot_ids is None:
        exclude_set: Set[str] = set()
    elif isinstance(exclude_bot_ids, str):
        exclude_set = {exclude_bot_ids}
    else:
        try:
            exclude_set = set(exclude_bot_ids)
        except Exception:
            exclude_set = set()

    client = WebClient(token=token)

    tz = ZoneInfo("America/New_York")
    now = run_dt or datetime.now(tz)
    if now.tzinfo is None:
        now = now.replace(tzinfo=tz)

    oldest, latest, start_local, end_local = _slack_window_for_daily_job(run_dt=now, tz=tz)

    if debug:
        print(f"[slack] channel={channel_id}")
        print(f"[slack] window: {start_local.isoformat()} -> {end_local.isoformat()}")
        if exclude_set:
            print(f"[slack] excluding bot_ids: {sorted(exclude_set)}")

    noisy_subtypes = {
        "bot_add",
        "channel_join",
        "channel_leave",
        "channel_topic",
        "channel_purpose",
        "channel_name",
        "channel_archive",
        "channel_unarchive",
    }

    messages: List[SlackMessage] = []
    cursor: Optional[str] = None

    try:
        while True:
            resp = client.conversations_history(
                channel=channel_id,
                oldest=str(oldest),
                latest=str(latest),
                limit=200,
                cursor=cursor,
                inclusive=True,
            )
            raw = resp.get("messages", []) or []
            if debug:
                print(f"[slack] fetched {len(raw)} messages (cursor={cursor!r})")

            for m in raw:
                subtype = m.get("subtype")
                if subtype in noisy_subtypes:
                    continue

                ts = m.get("ts")
                if not ts:
                    continue

                bot_id = m.get("bot_id")
                if bot_id and bot_id in exclude_set:
                    if debug:
                        print(f"[slack] skipping message ts={m.get('ts')} from excluded bot_id={bot_id}")
                    continue

                best_text = _extract_best_effort_text(m).strip()
                if not best_text:
                    continue

                messages.append(
                    SlackMessage(
                        ts=ts,
                        user_id=m.get("user"),
                        username=None,
                        text=best_text,
                        subtype=subtype,
                        bot_id=bot_id,
                    )
                )

                if len(messages) >= max_messages:
                    break

            if len(messages) >= max_messages:
                break

            cursor = (resp.get("response_metadata") or {}).get("next_cursor") or None
            if not cursor:
                break

    except SlackApiError as e:
        if debug:
            data = getattr(e, "response", None).data if getattr(e, "response", None) else None
            print("[slack] conversations_history failed:", data or str(e))
        return []

    messages.sort(key=lambda x: float(x.ts))
    return messages[:max_messages]


def _format_slack_feedback_block(
    messages: List[SlackMessage],
    *,
    resolve_usernames: bool = False,
    slack_api_key: Optional[str] = None,
    max_chars: int = 6000,
) -> str:
    """
    Formats Slack messages into a context block labeled as team feedback.
    If resolve_usernames=True, attempts users_info lookups (requires users:read).
    """
    if not messages:
        return ""

    import os
    from dotenv import load_dotenv

    load_dotenv()
    token = slack_api_key or os.getenv("SLACK_API_KEY")

    client = None
    user_cache: Dict[str, str] = {}

    if resolve_usernames and token:
        try:
            from slack_sdk import WebClient
            client = WebClient(token=token)
        except Exception:
            client = None

    def name_for(msg: SlackMessage) -> str:
        if msg.username:
            return msg.username
        if msg.user_id and client:
            if msg.user_id in user_cache:
                return user_cache[msg.user_id]
            try:
                u = client.users_info(user=msg.user_id)
                prof = (u.get("user") or {}).get("profile") or {}
                nm = prof.get("display_name") or prof.get("real_name") or (u.get("user") or {}).get("name") or msg.user_id
                user_cache[msg.user_id] = nm
                return nm
            except Exception:
                user_cache[msg.user_id] = msg.user_id
                return msg.user_id

        if msg.bot_id:
            return f"bot:{msg.bot_id}"
        return msg.user_id or "unknown"

    tz = ZoneInfo("America/New_York")

    lines: List[str] = []
    for m in messages:
        dt = datetime.fromtimestamp(float(m.ts), tz=tz)
        ts_str = dt.strftime("%Y-%m-%d %I:%M %p %Z")
        who = name_for(m)
        lines.append(f"- [{ts_str}] {who}: {m.text.replace('\\n', ' ').strip()}")

    block = "===== SLACK FEEDBACK (Team callouts for today's reporting) =====\n" + "\n".join(lines)
    if len(block) > max_chars:
        block = block[:max_chars] + "\n... (truncated)"
    return block

def get_prompt_context(
    *,
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    include_business_context: bool = True,
    include_glossary: bool = True,
    include_analyst_data_context: bool = True,
    include_vector_blocks: bool = True,
    max_vector_blocks: int = 8,
    max_chars_per_vector_block: int = 4000,
    include_slack_feedback: bool = True,
    slack_channel_id: Optional[str] = "C0ACEKFDH70",
    slack_api_key: Optional[str] = None,
    run_dt: Optional[datetime] = None,
    # optional controls you may want later without breaking call sites
    slack_max_messages: int = 400,
    slack_resolve_usernames: bool = False,
    slack_block_max_chars: int = 6000,
    slack_debug: bool = False,
) -> str:
    """
    Aggregates all prompt context into a single string.

    Compatible with the call pattern you showed:
      shared_context = get_prompt_context(
          retrieved_blocks=retrieved_blocks,
          include_business_context=True,
          include_glossary=True,
          include_analyst_data_context=True,
          include_vector_blocks=True,
          max_vector_blocks=...,
          max_chars_per_vector_block=4000,
          include_slack_feedback=True,
          slack_channel_id="C0ACY9XB82E",
      )
    """
    parts: List[str] = []

    # ----------------------------
    # 1) Static business context
    # ----------------------------
    if include_business_context:
        txt = (BUSINESS_CONTEXT_OVERVIEW or "").strip()
        if txt:
            parts.append(txt)

    if include_glossary:
        txt = (BUSINESS_CONTEXT_GLOSSARY or "").strip()
        if txt:
            parts.append(txt)

    if include_analyst_data_context:
        txt = (ANALYST_DATA_CONTEXT or "").strip()
        if txt:
            parts.append(txt)

    # ----------------------------
    # 2) Vector retrieval blocks
    # ----------------------------
    if include_vector_blocks and retrieved_blocks:
        vb = _format_vector_blocks(
            retrieved_blocks,
            max_blocks=max_vector_blocks,
            max_chars_per_block=max_chars_per_vector_block,
        ).strip()
        if vb:
            parts.append("===== VECTOR CONTEXT (Retrieved snippets) =====\n" + vb)

    # ----------------------------
    # 3) Slack feedback
    # ----------------------------
    if include_slack_feedback and slack_channel_id:
        msgs = _fetch_slack_messages_last_window(
            channel_id=slack_channel_id,
            slack_api_key=slack_api_key,
            run_dt=run_dt,
            max_messages=slack_max_messages,
            debug=slack_debug,
        )
        slack_block = _format_slack_feedback_block(
            msgs,
            resolve_usernames=slack_resolve_usernames,
            slack_api_key=slack_api_key,
            max_chars=slack_block_max_chars,
        ).strip()
        if slack_block:
            parts.append(slack_block)

    # Final assembly: double newlines keep sections visually separated
    return "\n\n".join([p for p in parts if p and p.strip()])


### Executive Summary

In [0]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional

from openai import OpenAI


def _top_block_texts(retrieved_blocks: Optional[Dict[str, Any]], max_blocks: int = 8) -> str:
    if not retrieved_blocks:
        return ""
    blocks = retrieved_blocks.get("blocks", {})
    try:
        ordered = sorted(blocks.values(), key=lambda b: int(b.get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())

    texts: List[str] = []
    for b in ordered[:max_blocks]:
        t = (b.get("text") or "").strip()
        if t:
            texts.append(t if len(t) <= 2000 else t[:2000] + " ...")
    return "\n\n".join(f"BLOCK[{i+1}]: {txt}" for i, txt in enumerate(texts))


def _extract_response_text(resp: Any) -> str:
    if resp is None:
        return ""

    t = getattr(resp, "output_text", None)
    if isinstance(t, str) and t.strip():
        return t.strip()

    out = getattr(resp, "output", None)
    if isinstance(out, list):
        chunks: List[str] = []
        for item in out:
            content = getattr(item, "content", None)
            if not isinstance(content, list):
                continue
            for part in content:
                ptype = getattr(part, "type", None) if not isinstance(part, dict) else part.get("type")
                if ptype in ("output_text", "text"):
                    ptext = getattr(part, "text", None) if not isinstance(part, dict) else part.get("text")
                    if ptext:
                        chunks.append(str(ptext))
        if chunks:
            return "\n".join(chunks).strip()

    try:
        return str(resp.output[0].content[0].text).strip()
    except Exception:
        pass

    if hasattr(resp, "model_dump"):
        try:
            d = resp.model_dump()
            if isinstance(d, dict) and isinstance(d.get("output_text"), str):
                return d["output_text"].strip()
        except Exception:
            pass

    return ""


def _get_tokenizer(model: str):
    """
    Best-effort tokenizer. Uses tiktoken if available; otherwise falls back to a rough heuristic.
    """
    try:
        import tiktoken  # type: ignore

        # tiktoken doesn't know every new model name; fall back safely.
        try:
            enc = tiktoken.encoding_for_model(model)
        except Exception:
            enc = tiktoken.get_encoding("o200k_base")
        return ("tiktoken", enc)
    except Exception:
        return ("heuristic", None)


def _count_tokens(text: str, tokenizer_kind: str, enc: Any) -> int:
    if not text:
        return 0
    if tokenizer_kind == "tiktoken" and enc is not None:
        return len(enc.encode(text))
    # heuristic: ~4 chars per token (very rough, but safe for truncation)
    return max(1, len(text) // 4)


def _truncate_to_tokens(text: str, max_tokens: int, tokenizer_kind: str, enc: Any) -> str:
    if max_tokens <= 0 or not text:
        return ""
    if _count_tokens(text, tokenizer_kind, enc) <= max_tokens:
        return text

    if tokenizer_kind == "tiktoken" and enc is not None:
        toks = enc.encode(text)
        toks = toks[:max_tokens]
        out = enc.decode(toks)
        return out + " ..."

    # heuristic truncation by chars
    approx_chars = max_tokens * 4
    return (text[:approx_chars] + " ...") if len(text) > approx_chars else text
    
def generate_executive_summary_for_report_text(
    *,
    kpi_json: Optional[Dict[str, Any]] = None,
    weekly_context: Optional[Dict[str, Any]] = None,
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    report_docx_path: Optional[str | "Path"] = None,
    model: str = "gpt-5",
    max_retrieved_blocks_in_prompt: int = 12,
    max_output_tokens: int = 10000,
    model_context_window_tokens: int = 256_000,
    safety_margin_tokens: int = 2_000,
    input_budget_fraction: float = 0.98,
    report_excerpt_char_limit: int = 200_000,
    debug: bool = True,
) -> str:
    """
    Executive summary generator using shared get_prompt_context(...) plus weekly + KPI + instructions
    and an optional report DOCX excerpt. Returns plain text only.
    """
    from pathlib import Path
    import re
    from datetime import datetime
    from docx import Document

    client = OpenAI()
    tokenizer_kind, enc = _get_tokenizer(model)

    # ----------------------------
    # Optional report DOCX extract
    # ----------------------------
    report_text = ""
    date_key: Optional[str] = None
    if report_docx_path:
        rp = Path(report_docx_path).resolve()
        if not rp.exists():
            raise FileNotFoundError(f"Report not found: {rp}")

        doc_in = Document(str(rp))
        parts: List[str] = []
        for p in doc_in.paragraphs:
            t = (p.text or "").strip()
            if t:
                parts.append(t)
        report_text = "\n".join(parts)

        try:
            _regex = globals().get("_REPORT_DATE_RE")
            if _regex is None or not hasattr(_regex, "search"):
                raise NameError
            REPORT_DATE_RE = _regex
        except Exception:
            REPORT_DATE_RE = re.compile(r"(?P<yyyy>\d{4})[_-](?P<mm>\d{2})[_-](?P<dd>\d{2})")

        m = REPORT_DATE_RE.search(rp.name)
        if m:
            try:
                dt = datetime(int(m["yyyy"]), int(m["mm"]), int(m["dd"]))
            except Exception:
                dt = datetime.now()
        else:
            dt = datetime.now()
        date_key = dt.strftime("%m_%d_%Y")

    # ----------------------------
    # Prompt-specific instructions
    # ----------------------------
    user_instructions = """
You are an experienced operations analyst writing a senior-leadership executive summary for Energy Voice.

Write a clear, narrative executive summary (2-3 short paragraphs) that walks through yesterday's performance
from top-of-funnel to revenue outcome.

Guidance:
- Start with a high-level assessment of the day (better / worse / mixed vs recent trend) in the first paragraph, outlining the most important metrics versus baseline. These metrics include Gross Call Volume, Queue to Gross, Queue/Net Call Counts, Net Conversion, Gross Conversion, RPGC, RPNC, RPO, and Revenue.
- In the next paragraph, outline Compass/IVR health specifically. Start with the initial question, where the bot rate will have an impact. Bots get filtered out here; it can look like breakage but is expected. Then cover notable breakage/queue rates for later steps. Wrap with queue call counts and queue-to-gross. Mention marketing buckets only if materially off baseline.
- If a third paragraph is needed, cover the net side: abandon rate, net call counts, calls into credit/contact, net conversion, RPO, and RPNC—again mentioning buckets only when materially different.
- Use specific metrics only where they materially support the story. Do NOT list metrics mechanically.

What to avoid:
- Do NOT mention call routing, routing logic, or queue routing behavior.
- Do NOT call out specific partner changes, partner-level shifts, or partner performance.

Tone and style:
- Brief a VP/GM: signal, not a dashboard readout.
- 2-3 short paragraphs, ~6-10 sentences total.
- No bullet points, no headings, no markdown.
- Return ONLY the executive summary text.
""".strip()

    # ----------------------------
    # Shared context (global)
    # ----------------------------
    shared_context = get_prompt_context(
        retrieved_blocks=retrieved_blocks,
        include_business_context=True,
        include_glossary=True,
        include_analyst_data_context=True,
        include_vector_blocks=True,
        max_vector_blocks=max_retrieved_blocks_in_prompt,
        max_chars_per_vector_block=4000,
        include_slack_feedback=True,
        slack_channel_id="C0ACEKFDH70", 
    )

    # ----------------------------
    # Data blocks (weekly + KPI)
    # ----------------------------
    kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, indent=2)
    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, indent=2)

    weekly_block = "\n\n".join(
        [
            "===== WEEKLY CONTEXT (prior daily summaries JSON) =====",
            weekly_text,
        ]
    ).strip()

    kpi_block = "\n\n".join(
        [
            "===== KPI JSON (yesterday) =====",
            kpi_text,
        ]
    ).strip()

    report_block = ""
    if report_text:
        excerpt = report_text[:report_excerpt_char_limit]
        report_block = "===== REPORT TEXT (excerpt) =====\n\"\"\"\n" + excerpt + "\n\"\"\""

    # ----------------------------
    # Token budgeting
    # ----------------------------
    hard_reserved = int(max_output_tokens) + int(safety_margin_tokens)
    remaining_for_input = max(0, int(model_context_window_tokens) - hard_reserved)
    input_budget = int(remaining_for_input * float(input_budget_fraction))

    def assemble(sc: str, wk: str, kpi_b: str, instr: str, rpt: str) -> str:
        parts: List[str] = []
        if date_key:
            parts.append(f"Report date: {date_key}")
        if sc:
            parts.append(sc)
        if wk:
            parts.append(wk)
        if kpi_b:
            parts.append(kpi_b)
        parts.append("===== INSTRUCTIONS =====\n" + instr)
        if rpt:
            parts.append(rpt)
        return "\n\n".join(parts).strip()

    full_prompt = assemble(shared_context, weekly_block, kpi_block, user_instructions, report_block)
    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        rpt_fixed = assemble(shared_context, weekly_block, kpi_block, user_instructions, "")
        fixed_tokens = _count_tokens(rpt_fixed, tokenizer_kind, enc)
        rpt_allow = max(0, input_budget - fixed_tokens)
        report_block = _truncate_to_tokens(report_block, rpt_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, user_instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble("", weekly_block, kpi_block, user_instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        sc_allow = max(0, input_budget - fixed_tokens)
        shared_context = _truncate_to_tokens(shared_context, sc_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, user_instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, "", kpi_block, user_instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        wk_allow = max(0, input_budget - fixed_tokens)
        weekly_block = _truncate_to_tokens(weekly_block, wk_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, user_instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, weekly_block, "", user_instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        kpi_allow = max(0, input_budget - fixed_tokens)
        kpi_block = _truncate_to_tokens(kpi_block, kpi_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, user_instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if debug:
        print("DEBUG: tokenizer =", tokenizer_kind)
        print("DEBUG: model_context_window_tokens =", model_context_window_tokens)
        print("DEBUG: max_output_tokens =", max_output_tokens)
        print("DEBUG: safety_margin_tokens =", safety_margin_tokens)
        print("DEBUG: input_budget_tokens =", input_budget)
        print("DEBUG: final_prompt_tokens =", total_tokens)
        print("DEBUG: shared_context chars =", len(shared_context))
        print("DEBUG: weekly_context chars =", len(weekly_text))
        print("DEBUG: kpi_json chars =", len(kpi_text))
        if report_text:
            print("DEBUG: report_text chars =", len(report_text))
            print("DEBUG: report_excerpt_char_limit =", report_excerpt_char_limit)

    # ----------------------------
    # CHANGED: medium reasoning + 4000 output tokens
    # ----------------------------
    resp = client.responses.create(
        model=model,
        input=full_prompt,
        max_output_tokens=max_output_tokens,
        reasoning={"effort": "medium"},
        text={"format": {"type": "text"}},
    )

    out_text = _extract_response_text(resp)

    if debug:
        status = getattr(resp, "status", None)
        types = [getattr(x, "type", None) for x in (getattr(resp, "output", None) or [])]
        print("DEBUG: status:", status)
        print("DEBUG: output item types:", types)
        if not out_text:
            print("DEBUG: EMPTY TEXT (likely only reasoning). incomplete_details:", getattr(resp, "incomplete_details", None))

    return out_text


### Drivers/Deep-Dive

In [0]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional

from openai import OpenAI


def _top_block_texts(retrieved_blocks: Optional[Dict[str, Any]], max_blocks: int = 8) -> str:
    """
    Return a compact concatenation of the top N retrieved blocks' text for inclusion in the prompt.
    Expects retrieved_blocks in the format returned by `retrieve_vector_store_blocks`:
      {"blocks": {"1": {...}, "2": {...}, ...}}
    """
    if not retrieved_blocks:
        return ""
    blocks = retrieved_blocks.get("blocks", {})
    try:
        ordered = sorted(blocks.values(), key=lambda b: int(b.get("block_id", 0)))
    except Exception:
        ordered = list(blocks.values())
    texts: List[str] = []
    for b in ordered[:max_blocks]:
        t = b.get("text", "").strip()
        if t:
            texts.append(t if len(t) <= 2000 else t[:2000] + " ...")
    return "\n\n".join(f"BLOCK[{i+1}]: {txt}" for i, txt in enumerate(texts))


# ---- Token helpers (best-effort) ----
def _get_tokenizer(model: str):
    """
    Best-effort tokenizer. Uses tiktoken if available; otherwise falls back to a rough heuristic.
    """
    try:
        import tiktoken  # type: ignore

        try:
            enc = tiktoken.encoding_for_model(model)
        except Exception:
            enc = tiktoken.get_encoding("o200k_base")
        return ("tiktoken", enc)
    except Exception:
        return ("heuristic", None)


def _count_tokens(text: str, tokenizer_kind: str, enc: Any) -> int:
    if not text:
        return 0
    if tokenizer_kind == "tiktoken" and enc is not None:
        return len(enc.encode(text))
    # heuristic: ~4 characters per token (rough)
    return max(1, len(text) // 4)


def _truncate_to_tokens(text: str, max_tokens: int, tokenizer_kind: str, enc: Any) -> str:
    """
    Truncate `text` to approximately `max_tokens` tokens.
    Returns an ellipsized string if truncated.
    """
    if max_tokens <= 0 or not text:
        return ""
    if _count_tokens(text, tokenizer_kind, enc) <= max_tokens:
        return text

    if tokenizer_kind == "tiktoken" and enc is not None:
        toks = enc.encode(text)
        toks = toks[:max_tokens]
        out = enc.decode(toks)
        return out + " ..."
    # fallback: char-based truncation
    approx_chars = max_tokens * 4
    return (text[:approx_chars] + " ...") if len(text) > approx_chars else text
    
def generate_step_level_drivers(
    *,
    kpi_json: Optional[Dict[str, Any]] = None,
    weekly_context: Optional[Dict[str, Any]] = None,
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    report_docx_path: Optional[str | "Path"] = None,
    model: str = "gpt-5",
    min_drivers: int = 3,
    max_retrieved_blocks_in_prompt: int = 12,
    # CHANGED: bigger output budget for ~2x exec summary
    max_output_tokens: int = 25000,
    model_context_window_tokens: int = 256_000,
    safety_margin_tokens: int = 3_000,
    input_budget_fraction: float = 0.98,
    report_excerpt_char_limit: int = 80_000,
    debug: bool = False,
) -> Dict[str, Any]:
    """
    Step-level drivers generator that uses shared `get_prompt_context(...)` context,
    plus WEEKLY_CONTEXT + YDAY_KPI_JSON + optional REPORT TEXT excerpt from a DOCX.

    Output is plain text with at least `min_drivers` drivers and a consistent structure:
      - Callout
      - Down-funnel impact
      - Evidence

    Optionally includes SMALL / SUSPICIOUS CHANGES when relevant; omitted if nothing meaningful.
    """
    from pathlib import Path
    import re
    from datetime import datetime
    from docx import Document

    client = OpenAI()
    tokenizer_kind, enc = _get_tokenizer(model)

    # ----------------------------
    # Optional report DOCX extract
    # ----------------------------
    report_text = ""
    date_key: Optional[str] = None
    if report_docx_path:
        rp = Path(report_docx_path).resolve()
        if not rp.exists():
            raise FileNotFoundError(f"Report not found: {rp}")

        doc_in = Document(str(rp))
        parts: List[str] = []
        for p in doc_in.paragraphs:
            t = (p.text or "").strip()
            if t:
                parts.append(t)
        report_text = "\n".join(parts)

        try:
            _regex = globals().get("_REPORT_DATE_RE")
            if _regex is None or not hasattr(_regex, "search"):
                raise NameError
            REPORT_DATE_RE = _regex
        except Exception:
            REPORT_DATE_RE = re.compile(r"(?P<yyyy>\d{4})[_-](?P<mm>\d{2})[_-](?P<dd>\d{2})")

        m = REPORT_DATE_RE.search(rp.name)
        if m:
            try:
                dt = datetime(int(m["yyyy"]), int(m["mm"]), int(m["dd"]))
            except Exception:
                dt = datetime.now()
        else:
            dt = datetime.now()
        date_key = dt.strftime("%m_%d_%Y")

    # ----------------------------
    # Prompt-specific instructions
    # ----------------------------
    instructions = f"""
TASK:
Identify the {min_drivers} main drivers (step-level or volume-source metrics) that explain yesterday's performance.
These should be the overall changes that impacted the flow the most. After the main ones, you may mention additional
marketing bucket / site vs SERP items ONLY if they materially explain movement.

CRITICAL OUTPUT FORMAT (PLAIN TEXT ONLY):
- Do NOT return JSON.
- Do NOT include a "Next steps" section anywhere.
- Write at least {min_drivers} drivers.
- Use this exact structure for each driver:

DRIVER <N>: <short title>
Callout: <what changed, stated plainly>
Down-funnel impact: <describe downstream impacts through the funnel: volume -> Compass filtering/first-question behavior/bot rate/breakage/queue -> agent call -> conversion -> revenue quality>
Evidence: <cite the specific KPI fields / deltas from the provided KPI JSON; be concrete>

Then, after you finish the main drivers, optionally add a section ONLY if applicable:

SMALL / SUSPICIOUS CHANGES (optional):
- Use 1–5 short bullets.
- These are low-volume but high-delta or isolated changes that might indicate something is wrong (often Compass flow).
- Write conversationally like: "One weird thing: ...", followed by the evidence from data.
- If there aren't meaningful items, omit this entire section.

IMPORTANT GUIDANCE:
- Treat missing KPI fields as intentionally omitted (insignificant) unless their absence materially changes confidence.
- Think in funnel flow: volume -> Compass health (first-question behavior, bot rate, breakage/queue rates) -> agent health (abandon, CIContact, CICredit, net calls, conversion, RPGC/RPO).
- These should be single drivers (one at a time), not multi-driver mashups.
- Prioritize Compass health first, then agent health.
- Avoid call-routing, routing logic, partner-level commentary.
""".strip()

    # ----------------------------
    # Shared context (global)
    # ----------------------------
    shared_context = get_prompt_context(
        retrieved_blocks=retrieved_blocks,
        include_business_context=True,
        include_glossary=True,
        include_analyst_data_context=True,
        include_vector_blocks=True,
        max_vector_blocks=max_retrieved_blocks_in_prompt,
        max_chars_per_vector_block=4000,
        include_slack_feedback=True,
        slack_channel_id="C0ACEKFDH70", 
    )

    # ----------------------------
    # Data blocks (weekly + KPI)
    # ----------------------------
    kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, separators=(",", ":"))
    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, separators=(",", ":"))

    weekly_block = "\n\n".join(["===== WEEKLY CONTEXT (prior daily summaries JSON) =====", weekly_text]).strip()
    kpi_block = "\n\n".join(["===== KPI JSON (yesterday) =====", kpi_text]).strip()

    report_block = ""
    if report_text:
        excerpt = report_text[:report_excerpt_char_limit]
        report_block = "===== REPORT TEXT (excerpt) =====\n\"\"\"\n" + excerpt + "\n\"\"\""

    # ----------------------------
    # Token budgeting
    # ----------------------------
    hard_reserved = int(max_output_tokens) + int(safety_margin_tokens)
    remaining_for_input = max(0, int(model_context_window_tokens) - hard_reserved)
    input_budget = int(remaining_for_input * float(input_budget_fraction))

    def assemble(sc: str, wk: str, kpi_b: str, instr: str, rpt: str) -> str:
        parts: List[str] = []
        if date_key:
            parts.append(f"Report date: {date_key}")
        parts.append("===== INSTRUCTIONS =====\n" + instr)
        if kpi_b:
            parts.append(kpi_b)
        if wk:
            parts.append(wk)
        if sc:
            parts.append(sc)
        if rpt:
            parts.append(rpt)
        return "\n\n".join([p for p in parts if p and p.strip()]).strip()

    full_prompt = assemble(shared_context, weekly_block, kpi_block, instructions, report_block)
    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, weekly_block, kpi_block, instructions, "")
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        rpt_allow = max(0, input_budget - fixed_tokens)
        report_block = _truncate_to_tokens(report_block, rpt_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble("", weekly_block, kpi_block, instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        sc_allow = max(0, input_budget - fixed_tokens)
        shared_context = _truncate_to_tokens(shared_context, sc_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, "", kpi_block, instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        wk_allow = max(0, input_budget - fixed_tokens)
        weekly_block = _truncate_to_tokens(weekly_block, wk_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, weekly_block, "", instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        kpi_allow = max(0, input_budget - fixed_tokens)
        kpi_block = _truncate_to_tokens(kpi_block, kpi_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if debug:
        print("DEBUG: tokenizer =", tokenizer_kind)
        print("DEBUG: model_context_window_tokens =", model_context_window_tokens)
        print("DEBUG: max_output_tokens =", max_output_tokens)
        print("DEBUG: safety_margin_tokens =", safety_margin_tokens)
        print("DEBUG: input_budget_tokens =", input_budget)
        print("DEBUG: final_prompt_tokens =", total_tokens)
        print("DEBUG: shared_context chars =", len(shared_context))
        print("DEBUG: weekly_context chars =", len(weekly_text))
        print("DEBUG: kpi_json chars =", len(kpi_text))
        if report_text:
            print("DEBUG: report_text chars =", len(report_text))
            print("DEBUG: report_excerpt_char_limit =", report_excerpt_char_limit)

    # ----------------------------
    # Model call (CHANGED: medium reasoning + explicit text format)
    # ----------------------------
    resp = client.responses.create(
        model=model,
        input=full_prompt,
        max_output_tokens=max_output_tokens,
        reasoning={"effort": "medium"},
        text={"format": {"type": "text"}},
    )

    raw = _extract_response_text(resp)

    if debug:
        status = getattr(resp, "status", None)
        types = [getattr(x, "type", None) for x in (getattr(resp, "output", None) or [])]
        print("DEBUG: status:", status)
        print("DEBUG: output item types:", types)
        if not raw:
            print("DEBUG: EMPTY TEXT. incomplete_details:", getattr(resp, "incomplete_details", None))

    # Defensive: enforce at least min_drivers
    def _count_driver_headers(txt: str) -> int:
        import re as _re
        return len(_re.findall(r"(?im)^\s*DRIVER\s+\d+\s*:", txt))

    if _count_driver_headers(raw) < min_drivers:
        nudge = f"""
You returned fewer than {min_drivers} drivers. Please rewrite the output as plain text with at least {min_drivers} drivers.
Follow the exact structure:

DRIVER <N>: <short title>
Callout: ...
Down-funnel impact: ...
Evidence: ...

Optionally add SMALL / SUSPICIOUS CHANGES (optional) at the end only if meaningful. No next steps. No JSON.
""".strip()

        resp2 = client.responses.create(
            model=model,
            input=full_prompt + "\n\n===== REVISION REQUEST =====\n" + nudge,
            max_output_tokens=max_output_tokens,
            reasoning={"effort": "medium"},
            text={"format": {"type": "text"}},
        )

        raw2 = _extract_response_text(resp2)
        if raw2:
            raw = raw2

        if debug:
            status2 = getattr(resp2, "status", None)
            types2 = [getattr(x, "type", None) for x in (getattr(resp2, "output", None) or [])]
            print("DEBUG: revision status:", status2)
            print("DEBUG: revision output item types:", types2)
            if not raw2:
                print("DEBUG: REVISION EMPTY TEXT. incomplete_details:", getattr(resp2, "incomplete_details", None))

    return {"text": raw, "output": raw}


### Pacing Section

In [0]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any, Dict, Optional, Sequence, Tuple

from openai import OpenAI


# -----------------------------
# SQL builder: month-to-date pacing vs INTERNAL PLAN (only the fields you care about)
# -----------------------------
from typing import Any, Dict, Tuple

def build_tx_mtd_pacing_sql(*, include_social: str = "Yes") -> Tuple[str, Dict[str, Any]]:
    """
    Exact same logic as your validated query, but:
      - final select is filtered to yesterday (as-of date)
      - keeps ONLY performance_view IN ('Pacing','Internal Plan','vs_internal')
      - projects ONLY the metrics/features we care about (phone funnel subset)
    """
    sql = """
WITH
dedupe AS (
  SELECT DISTINCT *
  FROM energy_prod.energy.rpt_texas_daily_pacing
),

params AS (
  SELECT :include_social AS include_social
),

cte_base AS (
  SELECT
    tx.rpt_date AS date,
    tx.month,
    CASE WHEN tx.performance_view = 'Final' THEN 'Plan' ELSE tx.performance_view END AS performance_view,

    /* ---- SITE phone ---- */
    SUM(tx.site_queue_calls) * 1.0 / NULLIF(SUM(tx.sessions),0)            AS site_rr,
    SUM(tx.site_queue_calls) * 1.0 / NULLIF(SUM(tx.site_gross_calls),0)    AS site_q2g,
    SUM(tx.site_queue_calls)                                               AS site_queue_calls,
    SUM(tx.site_abandon_calls) * 1.0 / NULLIF(SUM(tx.site_queue_calls),0)  AS site_abandon_rate,
    SUM(tx.site_net_calls)                                                 AS site_net_calls,
    SUM(tx.site_phone_orders) * 1.0 / NULLIF(SUM(tx.site_net_calls),0)     AS site_conversion_rate,
    SUM(tx.site_phone_orders)                                              AS site_orders,

    /* ---- SERP phone ---- */
    SUM(tx.serp_queue_calls) * 1.0 / NULLIF(SUM(tx.clicks),0)              AS serp_rr,
    SUM(tx.serp_queue_calls) * 1.0 / NULLIF(SUM(tx.serp_gross_calls),0)    AS serp_q2g,
    SUM(tx.serp_queue_calls)                                               AS serp_queue_calls,
    SUM(tx.serp_abandon_calls) * 1.0 / NULLIF(SUM(tx.serp_queue_calls),0)  AS serp_abandon_rate,
    SUM(tx.serp_net_calls)                                                 AS serp_net_calls,
    SUM(tx.serp_orders) * 1.0 / NULLIF(SUM(tx.serp_net_calls),0)           AS serp_conversion_rate,
    SUM(tx.serp_orders)                                                    AS serp_orders,

    /* ---- TOTAL phone ---- */
    SUM(tx.total_queue_calls)                                              AS total_queue_calls,
    SUM(tx.total_abandon_calls) * 1.0 / NULLIF(SUM(tx.total_queue_calls),0) AS total_abandon_rate,
    SUM(tx.total_net_calls)                                                AS total_net_calls,
    SUM(tx.phone_orders) * 1.0 / NULLIF(SUM(tx.total_net_calls),0)          AS total_conversion_rate,
    SUM(tx.phone_orders)                                                   AS total_phone_orders,

    /* ---- Phone value ---- */
    SUM(tx.site_phone_orders) * 1.0 / NULLIF(SUM(tx.sessions),0)            AS phone_vc,
    SUM(tx.phone_revenue)                                                  AS phone_revenue

  FROM dedupe tx
  CROSS JOIN params p
  LEFT JOIN energy_prod.energy.date_dim dd ON tx.month = dd.full_date
  WHERE tx.performance_view IN ('Pacing','Final','Internal Plan')
    AND tx.rpt_date >= date_trunc('month', current_date - 1)
    AND tx.rpt_date <= current_date - 1
    AND CASE WHEN lower(p.include_social) = 'no' THEN tx.MarketingChannel != 'Social' ELSE true END
  GROUP BY 1,2,3
),

cte_delta_vs_internal AS (
  SELECT
    b.date,
    b.month,
    'vs_internal' AS performance_view,

    /* Rates: ratio - 1 (relative delta) */
    (MAX(b.site_rr)              / NULLIF(MAX(ip.site_rr),0))              - 1 AS site_rr,
    (MAX(b.site_q2g)             / NULLIF(MAX(ip.site_q2g),0))             - 1 AS site_q2g,
    (MAX(b.site_abandon_rate)    / NULLIF(MAX(ip.site_abandon_rate),0))    - 1 AS site_abandon_rate,
    (MAX(b.site_conversion_rate) / NULLIF(MAX(ip.site_conversion_rate),0)) - 1 AS site_conversion_rate,

    (MAX(b.serp_rr)              / NULLIF(MAX(ip.serp_rr),0))              - 1 AS serp_rr,
    (MAX(b.serp_q2g)             / NULLIF(MAX(ip.serp_q2g),0))             - 1 AS serp_q2g,
    (MAX(b.serp_abandon_rate)    / NULLIF(MAX(ip.serp_abandon_rate),0))    - 1 AS serp_abandon_rate,
    (MAX(b.serp_conversion_rate) / NULLIF(MAX(ip.serp_conversion_rate),0)) - 1 AS serp_conversion_rate,

    (MAX(b.total_abandon_rate)   / NULLIF(MAX(ip.total_abandon_rate),0))   - 1 AS total_abandon_rate,
    (MAX(b.total_conversion_rate)/ NULLIF(MAX(ip.total_conversion_rate),0)) - 1 AS total_conversion_rate,

    /* Volumes / dollars: ratio - 1 */
    (MAX(b.site_queue_calls)     / NULLIF(MAX(ip.site_queue_calls),0))     - 1 AS site_queue_calls,
    (MAX(b.site_net_calls)       / NULLIF(MAX(ip.site_net_calls),0))       - 1 AS site_net_calls,
    (MAX(b.site_orders)          / NULLIF(MAX(ip.site_orders),0))          - 1 AS site_orders,

    (MAX(b.serp_queue_calls)     / NULLIF(MAX(ip.serp_queue_calls),0))     - 1 AS serp_queue_calls,
    (MAX(b.serp_net_calls)       / NULLIF(MAX(ip.serp_net_calls),0))       - 1 AS serp_net_calls,
    (MAX(b.serp_orders)          / NULLIF(MAX(ip.serp_orders),0))          - 1 AS serp_orders,

    (MAX(b.total_queue_calls)    / NULLIF(MAX(ip.total_queue_calls),0))    - 1 AS total_queue_calls,
    (MAX(b.total_net_calls)      / NULLIF(MAX(ip.total_net_calls),0))      - 1 AS total_net_calls,
    (MAX(b.total_phone_orders)   / NULLIF(MAX(ip.total_phone_orders),0))   - 1 AS total_phone_orders,

    (MAX(b.phone_vc)             / NULLIF(MAX(ip.phone_vc),0))             - 1 AS phone_vc,
    (MAX(b.phone_revenue)        / NULLIF(MAX(ip.phone_revenue),0))        - 1 AS phone_revenue

  FROM cte_base b
  LEFT JOIN cte_base ip
    ON ip.date = b.date
   AND ip.month = b.month
   AND ip.performance_view = 'Internal Plan'
  WHERE b.performance_view = 'Pacing'
  GROUP BY 1,2,3
),

final_union AS (
  SELECT * FROM cte_base
  UNION ALL
  SELECT * FROM cte_delta_vs_internal
),

asof_date AS (
  SELECT
    CASE
      WHEN dayofmonth(current_date()) = 1 THEN last_day(add_months(current_date(), -1))
      ELSE date_sub(current_date(), 1)
    END AS d
)

SELECT
  fu.date,
  fu.month,
  fu.performance_view,

  fu.site_rr,
  fu.site_q2g,
  fu.site_queue_calls,
  fu.site_abandon_rate,
  fu.site_net_calls,
  fu.site_conversion_rate,
  fu.site_orders,

  fu.serp_rr,
  fu.serp_q2g,
  fu.serp_queue_calls,
  fu.serp_abandon_rate,
  fu.serp_net_calls,
  fu.serp_conversion_rate,
  fu.serp_orders,

  fu.total_queue_calls,
  fu.total_abandon_rate,
  fu.total_net_calls,
  fu.total_conversion_rate,
  fu.total_phone_orders,

  fu.phone_vc,
  fu.phone_revenue

FROM final_union fu
CROSS JOIN asof_date a
WHERE fu.date = a.d
  AND fu.performance_view IN ('Pacing','Internal Plan','vs_internal')
;
""".strip()

    return sql, {"include_social": include_social}

from __future__ import annotations

import json
from typing import Any, Dict, List, Optional, Tuple

from openai import OpenAI


# --------------------------------------------
# 1) DBX-native pacing payload (with real date)
# --------------------------------------------
def fetch_tx_mtd_pacing(*, include_social: str = "Yes") -> Dict[str, Any]:
    """
    Databricks-native pacing payload builder.

    - Runs the MTD pacing SQL via spark.sql
    - Returns an explicit `as_of_date` that matches the SQL window end (date_sub(current_date(), 1))
    """
    sql, _params = build_tx_mtd_pacing_sql(include_social=include_social)

    val = (include_social or "Yes").strip()
    if val.lower() not in ("yes", "no"):
        raise ValueError("include_social must be 'Yes' or 'No'")

    # spark.sql doesn't support named bind params like :include_social
    rendered_sql = sql.replace(":include_social", f"'{val}'")

    df = spark.sql(rendered_sql)
    print("MTD Pacing DF:")
    display(df)
    rows = [r.asDict(recursive=True) for r in df.collect()]

    by_view: Dict[str, Dict[str, Any]] = {}
    month_val: Optional[str] = None
    for r in rows:
        view = str(r.get("performance_view") or "").strip()
        if not view:
            continue
        month_val = month_val or str(r.get("month"))
        metrics = {k: r.get(k) for k in r.keys() if k not in ("month", "performance_view")}
        by_view[view] = metrics

    # Compute the exact as_of_date used in SQL (yesterday per warehouse clock)
    as_of_date = (
        spark.sql("SELECT date_sub(current_date(), 1) AS d")
        .collect()[0]["d"]
    )
    as_of_date_str = str(as_of_date)  # typically YYYY-MM-DD

    return {
        "month": month_val,
        "as_of_date": as_of_date_str,  # <-- real date string
        "include_social": val,
        "pacing": by_view.get("Pacing", {}),
        "internal_plan": by_view.get("Internal Plan", {}),
        "delta_vs_internal_plan": by_view.get("vs_internal_plan", {}),
    }


# --------------------------------------------
# 2) Robust extraction for Responses API text
# --------------------------------------------
def _extract_output_text(resp: Any) -> str:
    """
    Defensive: support multiple response shapes.
    Prefers `resp.output_text` when available; falls back to scanning output content.
    """
    txt = (getattr(resp, "output_text", None) or "").strip()
    if txt:
        return txt

    # Fallback: attempt to walk common structures
    out = getattr(resp, "output", None)
    if isinstance(out, list):
        chunks: List[str] = []
        for item in out:
            content = getattr(item, "content", None)
            if isinstance(content, list):
                for c in content:
                    # Common shapes: {"type":"output_text","text":"..."} or attributes
                    t = ""
                    if isinstance(c, dict):
                        t = str(c.get("text") or "")
                    else:
                        t = str(getattr(c, "text", "") or "")
                    if t:
                        chunks.append(t)
        return "\n".join(chunks).strip()

    return ""

def generate_mtd_pacing_narrative(
    *,
    weekly_context: Optional[Dict[str, Any]] = None,
    retrieved_blocks: Optional[Dict[str, Any]] = None,
    pacing_payload: Optional[Dict[str, Any]] = None,
    # optional extras (kept for compatibility)
    kpi_json: Optional[Dict[str, Any]] = None,
    report_docx_path: Optional[str | "Path"] = None,
    # model/runtime
    model: str = "gpt-5",
    max_retrieved_blocks_in_prompt: int = 12,
    max_output_tokens: int = 25_000,
    model_context_window_tokens: int = 256_000,
    safety_margin_tokens: int = 2_000,
    input_budget_fraction: float = 0.98,
    report_excerpt_char_limit: int = 30_000,
    kpi_char_limit: int = 25_000,
    debug: bool = False,
) -> Dict[str, Any]:
    """
    Writes a 1–2 paragraph "Month-to-date pacing" narrative.

    Updated: includes shared_context via get_prompt_context(...) the same way
    generate_executive_summary_for_report_text does.
    """
    from pathlib import Path
    import re
    from datetime import datetime
    from docx import Document

    client = OpenAI()
    tokenizer_kind, enc = _get_tokenizer(model)

    # ----------------------------
    # Validate pacing payload
    # ----------------------------
    if not pacing_payload or not isinstance(pacing_payload, dict) or not pacing_payload.get("pacing"):
        raise ValueError("pacing_payload is missing or empty. It must include at least a 'pacing' dict.")

    # ----------------------------
    # Optional report DOCX extract (lowest priority)
    # ----------------------------
    report_text = ""
    date_key: Optional[str] = None
    if report_docx_path:
        rp = Path(report_docx_path).resolve()
        if not rp.exists():
            raise FileNotFoundError(f"Report not found: {rp}")

        doc_in = Document(str(rp))
        parts: List[str] = []
        for p in doc_in.paragraphs:
            t = (p.text or "").strip()
            if t:
                parts.append(t)
        report_text = "\n".join(parts)

        try:
            _regex = globals().get("_REPORT_DATE_RE")
            if _regex is None or not hasattr(_regex, "search"):
                raise NameError
            REPORT_DATE_RE = _regex
        except Exception:
            REPORT_DATE_RE = re.compile(r"(?P<yyyy>\d{4})[_-](?P<mm>\d{2})[_-](?P<dd>\d{2})")

        m = REPORT_DATE_RE.search(rp.name)
        if m:
            try:
                dt = datetime(int(m["yyyy"]), int(m["mm"]), int(m["dd"]))
            except Exception:
                dt = datetime.now()
        else:
            dt = datetime.now()
        date_key = dt.strftime("%m_%d_%Y")

    # ----------------------------
    # Shared context (global)  <-- NEW
    # ----------------------------
    # This mirrors the exec-summary pattern. It also means you don't need to separately
    # dump retrieved_blocks into the prompt; get_prompt_context can include vector blocks.
    shared_context = get_prompt_context(
        retrieved_blocks=retrieved_blocks,
        include_business_context=True,
        include_glossary=True,
        include_analyst_data_context=True,
        include_vector_blocks=True,
        max_vector_blocks=max_retrieved_blocks_in_prompt,
        max_chars_per_vector_block=4000,
        include_slack_feedback=True,
        slack_channel_id="C0ACEKFDH70",
    )

    # ----------------------------
    # Static metric context block
    # ----------------------------
    metrics_context = """
===== PACING METRICS CONTEXT =====
Everything is month-to-date (MTD) through as_of_date. Use delta_vs_internal_plan (ratio-1) as the comparison
(0.10 = +10%, -0.05 = -5%).

PHONE FUNNEL LENS (MTD):
- RR:
  * Site RR = site_queue_calls / sessions
  * SERP RR = serp_queue_calls / clicks
  If RR is down vs internal plan, queue calls and downstream net calls/orders typically fall.

- Q2G = queue_calls / gross_calls:
  If Q2G is down, more leakage occurs before queue (breakage/filtering/first-question/bot behavior).

- Queue Calls:
  Primary lever for net calls and orders volume.

- Abandon Rate = abandon_calls / queue_calls:
  Higher abandon suppresses net calls even if queue calls are stable.

- Net Calls:
  Post-abandon call volume; the “agent opportunity” pool.

- Conversion Rate = orders / net_calls:
  If down, orders and revenue follow.

- Orders:
  Final phone orders.

TOTALS:
- total_queue_calls / total_net_calls / total_abandon_rate / total_conversion_rate / total_phone_orders summarize overall phone health,
  especially when SERP and Site move differently.

VALUE:
- phone_vc = site_phone_orders / sessions
- phone_revenue = phone revenue MTD
""".strip()

    # ----------------------------
    # Data blocks
    # ----------------------------
    pacing_text = json.dumps(pacing_payload, ensure_ascii=False, separators=(",", ":"), default=str)
    pacing_block = "\n\n".join(
        ["===== MTD PACING PAYLOAD (Pacing vs Internal Plan) =====", pacing_text]
    ).strip()

    weekly_text = json.dumps(weekly_context or {}, ensure_ascii=False, separators=(",", ":"))
    weekly_block = "\n\n".join(
        ["===== WEEKLY CONTEXT (prior daily summaries JSON) =====", weekly_text]
    ).strip()

    kpi_block = ""
    if kpi_json:
        kpi_text = json.dumps(kpi_json or {}, ensure_ascii=False, separators=(",", ":"))
        if len(kpi_text) <= int(kpi_char_limit):
            kpi_block = "\n\n".join(["===== KPI JSON (yesterday, optional) =====", kpi_text]).strip()

    report_block = ""
    if report_text:
        excerpt = report_text[: int(report_excerpt_char_limit)]
        report_block = "===== REPORT TEXT (excerpt, optional) =====\n\"\"\"\n" + excerpt + "\n\"\"\""

    # ----------------------------
    # Instructions
    # ----------------------------
    instructions = """
TASK:
Write the "Month-to-date pacing" section interpreting business performance so far this month.

OUTPUT:
- Plain text only.
- 1–2 paragraphs (no bullets).
- Must explicitly state whether we’re ahead/behind Internal Plan MTD and where in the phone funnel the gap is.
- Use concrete evidence: cite the exact metric fields and whether they’re up/down vs internal plan
  using delta_vs_internal_plan where available.
- If something looks like a measurement / tracking issue (low volume but huge delta, isolated SERP/Site weirdness),
  add 1–2 short sentences at the end calling it out with evidence.
- No next steps. No JSON.
""".strip()

    # ----------------------------
    # Token budgeting
    # ----------------------------
    hard_reserved = int(max_output_tokens) + int(safety_margin_tokens)
    remaining_for_input = max(0, int(model_context_window_tokens) - hard_reserved)
    input_budget = int(remaining_for_input * float(input_budget_fraction))

    def assemble(
        sc: str,
        pacing_b: str,
        metrics_ctx: str,
        wk: str,
        kpi_b: str,
        instr: str,
        rpt: str,
    ) -> str:
        parts: List[str] = []
        if date_key:
            parts.append(f"Report date: {date_key}")

        # Match exec-summary style: shared context first, then the task-specific data.
        if sc:
            parts.append(sc)

        # Keep the pacing narrative’s “must be present” data prominent.
        parts.append(pacing_b)
        parts.append(metrics_ctx)
        if wk:
            parts.append(wk)
        if kpi_b:
            parts.append(kpi_b)

        parts.append("===== INSTRUCTIONS =====\n" + instr)

        # Lowest priority
        if rpt:
            parts.append(rpt)

        return "\n\n".join([p for p in parts if p and p.strip()]).strip()

    full_prompt = assemble(
        shared_context,
        pacing_block,
        metrics_context,
        weekly_block,
        kpi_block,
        instructions,
        report_block,
    )
    total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    # Trim order (lowest priority first): report -> KPI -> weekly -> shared_context -> metrics_context (last resort)
    if total_tokens > input_budget:
        fixed = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, "")
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        rpt_allow = max(0, input_budget - fixed_tokens)
        report_block = _truncate_to_tokens(report_block, rpt_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        # KPI is optional
        kpi_block = ""
        full_prompt = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble(shared_context, pacing_block, metrics_context, "", kpi_block, instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        wk_allow = max(0, input_budget - fixed_tokens)
        weekly_block = _truncate_to_tokens(weekly_block, wk_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        fixed = assemble("", pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        sc_allow = max(0, input_budget - fixed_tokens)
        shared_context = _truncate_to_tokens(shared_context, sc_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if total_tokens > input_budget:
        # last resort: truncate metrics context (keep something rather than failing)
        fixed = assemble(shared_context, pacing_block, "", weekly_block, kpi_block, instructions, report_block)
        fixed_tokens = _count_tokens(fixed, tokenizer_kind, enc)
        mc_allow = max(0, input_budget - fixed_tokens)
        metrics_context = _truncate_to_tokens(metrics_context, mc_allow, tokenizer_kind, enc)
        full_prompt = assemble(shared_context, pacing_block, metrics_context, weekly_block, kpi_block, instructions, report_block)
        total_tokens = _count_tokens(full_prompt, tokenizer_kind, enc)

    if debug:
        print("DEBUG(pacing): tokenizer =", tokenizer_kind)
        print("DEBUG(pacing): model_context_window_tokens =", model_context_window_tokens)
        print("DEBUG(pacing): max_output_tokens =", max_output_tokens)
        print("DEBUG(pacing): safety_margin_tokens =", safety_margin_tokens)
        print("DEBUG(pacing): input_budget_tokens =", input_budget)
        print("DEBUG(pacing): final_prompt_tokens =", total_tokens)
        print("DEBUG(pacing): shared_context chars =", len(shared_context))
        print("DEBUG(pacing): pacing_payload chars =", len(pacing_text))
        print("DEBUG(pacing): metrics_context chars =", len(metrics_context))
        print("DEBUG(pacing): weekly_context chars =", len(weekly_text))

    # ----------------------------
    # Model call with retry-on-empty
    # ----------------------------
    def _extract_text(resp: Any) -> str:
        # Prefer the more robust helper if it's in scope; otherwise fall back.
        try:
            return _extract_response_text(resp)  # type: ignore[name-defined]
        except Exception:
            return _extract_output_text(resp)

    def call(prompt: str, effort: str = "medium") -> str:
        resp = client.responses.create(
            model=model,
            input=prompt,
            max_output_tokens=max_output_tokens,
            reasoning={"effort": effort},
            text={"format": {"type": "text"}},
        )
        if debug:
            status = getattr(resp, "status", None)
            types = [getattr(x, "type", None) for x in (getattr(resp, "output", None) or [])]
            print("DEBUG(pacing) call status:", status)
            print("DEBUG(pacing) output item types:", types)
            if getattr(resp, "incomplete_details", None):
                print("DEBUG(pacing) incomplete_details:", getattr(resp, "incomplete_details", None))
        return _extract_text(resp)

    raw = call(full_prompt, effort="medium")

    if not raw:
        if debug:
            print("DEBUG(pacing): first attempt returned empty. Retrying with trimmed prompt (medium).")
        slim_weekly = _truncate_to_tokens(weekly_block, 500, tokenizer_kind, enc)
        slim_shared = _truncate_to_tokens(shared_context, 3000, tokenizer_kind, enc)
        slim_prompt = assemble(
            slim_shared,
            pacing_block,
            metrics_context,
            slim_weekly,
            "",  # drop KPI
            instructions,
            "",  # drop report
        )
        raw = call(slim_prompt, effort="medium")

    if not raw:
        if debug:
            print("DEBUG(pacing): second attempt returned empty. Final retry with minimal prompt (low effort).")
        minimal_prompt = "\n\n".join(
            [
                shared_context,
                pacing_block,
                metrics_context,
                "===== INSTRUCTIONS =====\n" + instructions,
            ]
        ).strip()
        raw = call(minimal_prompt, effort="low")

    if debug:
        print("DEBUG(pacing) final output length:", len(raw or ""))

    return {"text": (raw or "").strip(), "output": (raw or "").strip()}


# Exporting Report

### Concatenating Text

In [0]:
from __future__ import annotations

import ast
import json
import re
from typing import Any, Dict, List, Optional, Tuple


# -------------------------
# Robust JSON extraction + salvage (kept, but now only used if you still want it elsewhere)
# -------------------------
def _clean_text_for_json(raw: str) -> str:
    """Normalize common LLM artifacts without being overly destructive."""
    if not isinstance(raw, str):
        raw = str(raw)

    raw = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```\s*$", "", raw)

    raw = (
        raw.replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    raw = "".join(ch for ch in raw if (ord(ch) >= 32) or ch in "\n\r\t")
    return raw


def _basic_repairs(s: str) -> str:
    """Small repairs that often fix 'almost JSON' outputs."""
    s = re.sub(r",\s*(\}|])", r"\1", s)
    s = re.sub(r"\bNone\b", "null", s)
    s = re.sub(r"\bTrue\b", "true", s)
    s = re.sub(r"\bFalse\b", "false", s)
    return s


def extract_and_parse_json(raw: str) -> Tuple[Any, str]:
    """
    Parse JSON from a string that may contain extra text before/after the JSON.
    Returns: (parsed_obj, cleaned_json_substring_used)
    """
    s = _basic_repairs(_clean_text_for_json(raw)).strip()

    try:
        return json.loads(s), s
    except json.JSONDecodeError:
        pass

    dec = json.JSONDecoder()
    starts = [m.start() for m in re.finditer(r"[\{\[]", s)]
    for i in starts:
        try:
            obj, end = dec.raw_decode(s[i:])
            json_sub = s[i : i + end]
            return obj, json_sub
        except json.JSONDecodeError:
            continue

    try:
        py_obj = ast.literal_eval(s)
        if isinstance(py_obj, dict):
            for k in ("raw_model_output", "raw", "output", "raw_output"):
                inner = py_obj.get(k)
                if isinstance(inner, str):
                    return extract_and_parse_json(inner)
        return py_obj, s
    except Exception:
        pass

    raise ValueError("No valid JSON object/array found in input.")


# -------------------------
# Formatting helpers (drivers)
# -------------------------
def _format_driver(driver: Dict[str, Any], index: int) -> str:
    """
    Supports BOTH:
      - new plain-text drivers output: {"text": "..."} (handled in finalize_report; not here)
      - old JSON drivers output: {"drivers":[{...}]} (still supported for backward compatibility)
    """
    title = driver.get("title", "Untitled driver")
    what = driver.get("what_happened", driver.get("what", "No description provided."))
    what_it_drove = driver.get("what_it_drove", "Downstream impact not specified.")
    next_steps = driver.get("next_steps", []) or []
    evidence = driver.get("evidence", "none")

    lines: List[str] = []
    lines.append(f"{index}. {title}")
    lines.append(f"   What happened: {what}")
    lines.append(f"   Impact: {what_it_drove}")
    if next_steps:
        lines.append("   Next steps:")
        for i, step in enumerate(next_steps[:5], start=1):
            lines.append(f"     {i}) {step}")
    lines.append(f"   Evidence: {evidence}")
    return "\n".join(lines)


# -------------------------
# Final function: accept summary, drivers, pacing_result and return concatenated text
# -------------------------
def finalize_report(
    *,
    summary: str,
    drivers_payload: Any,
    pacing_result: Any,
    max_driver_items: Optional[int] = None,
) -> Dict[str, Any]:
    """
    summary: executive summary text (string)

    drivers_payload:
      - NEW: dict returned by generate_step_level_drivers -> {"text": "...", "output": "..."}
      - OLD (back-compat): dict with {"drivers": [...]} or raw model JSON/string

    pacing_result:
      - dict returned by generate_mtd_pacing_narrative -> {"text": "...", "output": "..."}
      - or raw string fallback

    Returns a dict:
      {
        "combined_text": "<summary + drivers + pacing human text>",
        "summary_text": summary,
        "drivers_text": "<drivers section text>",
        "pacing_text": "<pacing narrative text>"
      }
    """
    # -------------------------
    # 1) Drivers section
    # -------------------------
    drivers_text = ""

    # Preferred: new plain-text drivers output
    if isinstance(drivers_payload, dict) and isinstance(drivers_payload.get("text"), str):
        drivers_text = drivers_payload["text"].strip()

    # Back-compat: old structured drivers list
    elif isinstance(drivers_payload, dict) and isinstance(drivers_payload.get("drivers"), list):
        drivers_list = drivers_payload["drivers"]
        if max_driver_items is not None:
            drivers_list = drivers_list[:max_driver_items]
        driver_blocks: List[str] = []
        for i, d in enumerate(drivers_list, start=1):
            if isinstance(d, dict):
                driver_blocks.append(_format_driver(d, i))
            else:
                driver_blocks.append(f"{i}. Unstructured driver item: {d!r}")
        drivers_text = "\n\n".join(driver_blocks) if driver_blocks else "No drivers provided."

    # Last resort: stringify
    else:
        drivers_text = str(drivers_payload).strip() if drivers_payload is not None else "No drivers provided."

    if not drivers_text:
        drivers_text = "No drivers provided."

    # -------------------------
    # 2) Pacing section
    # -------------------------
    pacing_text = ""
    if isinstance(pacing_result, dict) and isinstance(pacing_result.get("text"), str):
        pacing_text = pacing_result["text"].strip()
    elif isinstance(pacing_result, str):
        pacing_text = pacing_result.strip()
    else:
        pacing_text = str(pacing_result).strip() if pacing_result is not None else ""

    if not pacing_text:
        pacing_text = "No pacing narrative provided."

    # -------------------------
    # 3) Combine into final report text
    # -------------------------
    combined_parts: List[str] = []

    if summary and str(summary).strip():
        combined_parts.append("=== EXECUTIVE SUMMARY ===")
        combined_parts.append(str(summary).strip())

    combined_parts.append("\n=== STEP-LEVEL DRIVERS ===")
    combined_parts.append(drivers_text)

    combined_parts.append("\n=== MONTH-TO-DATE PACING ===")
    combined_parts.append(pacing_text)

    combined_text = "\n\n".join(combined_parts).strip()

    return {
        "combined_text": combined_text,
        "summary_text": summary,
        "drivers_text": drivers_text,
        "pacing_text": pacing_text,
    }


### Send to Slack Channel

#### Saving Doc File & Uploading, Plus KPI csv

In [0]:
from __future__ import annotations

import os
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo


# -----------------------------
# Slack token + API helpers
# -----------------------------
def get_slack_token(env_file: str = ".env") -> str:
    load_dotenv(dotenv_path=env_file, override=True)
    token = os.getenv("SLACK_API_KEY") or os.getenv("SLACK_BOT_TOKEN")
    if not token:
        raise RuntimeError("Missing SLACK_API_KEY (or SLACK_BOT_TOKEN) in environment")
    return token


def _slack_api_call(
    endpoint: str,
    *,
    method: str = "POST",
    data=None,
    params=None,
    files=None,
    timeout_s: int = 30,
    env_file: str = ".env",
) -> Dict[str, Any]:
    import requests  # type: ignore

    token = get_slack_token(env_file=env_file)
    url = f"https://slack.com/api/{endpoint}"
    headers = {"Authorization": f"Bearer {token}"}

    resp = requests.request(method, url, headers=headers, data=data, params=params, files=files, timeout=timeout_s)
    j = resp.json()
    j["_http_status"] = resp.status_code
    return j


def _slack_api_call_json(
    endpoint: str,
    *,
    payload: dict,
    timeout_s: int = 30,
    env_file: str = ".env",
) -> Dict[str, Any]:
    import requests  # type: ignore

    token = get_slack_token(env_file=env_file)
    url = f"https://slack.com/api/{endpoint}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json; charset=utf-8",
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=timeout_s)
    j = resp.json()
    j["_http_status"] = resp.status_code
    return j


def _upload_bytes_via_external_flow(
    *,
    filename: str,
    content: bytes,
    channel_id: str,
    title: Optional[str] = None,
    timeout_s: int = 30,
    env_file: str = ".env",
) -> Dict[str, Any]:
    """
    Slack external upload flow:
      1) files.getUploadURLExternal
      2) POST multipart to upload_url
      3) files.completeUploadExternal

    Per request:
      - No initial_comment
      - Keep docx name as-is
      - Excel should be exactly 'daily_kpis.xlsx'
    """
    import requests  # type: ignore

    # 1) Get upload URL + file_id
    get_url = _slack_api_call(
        "files.getUploadURLExternal",
        data={"filename": filename, "length": str(len(content))},
        timeout_s=timeout_s,
        env_file=env_file,
    )
    if not get_url.get("ok"):
        raise RuntimeError(f"files.getUploadURLExternal failed: {get_url.get('error')}")

    upload_url = get_url["upload_url"]
    file_id = get_url["file_id"]

    # 2) POST multipart to upload_url
    post = requests.post(
        upload_url,
        files={"file": (filename, content)},
        data={"filename": filename},
        timeout=timeout_s,
    )
    if post.status_code // 100 != 2:
        raise RuntimeError(f"Upload POST failed HTTP {post.status_code}: {post.text[:300]!r}")

    # 3) Complete upload + share
    payload: Dict[str, Any] = {
        "files": [{"id": file_id, "title": title or filename}],
        "channel_id": channel_id,
    }

    complete = _slack_api_call_json(
        "files.completeUploadExternal",
        payload=payload,
        timeout_s=timeout_s,
        env_file=env_file,
    )
    if not complete.get("ok"):
        raise RuntimeError(f"files.completeUploadExternal failed: {complete.get('error')}")

    return complete


# -----------------------------
# Excel generation
# -----------------------------
CANON_COLS = ["sort_key", "KPI", "P4WA", "Yesterday", "Delta", "Call_Count_Delta"]


def _safe_sheet_title(name: str) -> str:
    # Excel constraints: <=31 chars, no : \ / ? * [ ]
    bad = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in bad:
        name = name.replace(ch, " ")
    name = " ".join(name.split())
    if not name:
        name = "Sheet"
    return name[:31]

def _coerce_rows_to_dicts(rows: Any) -> List[Dict[str, Any]]:
    """
    Accepts:
      - pandas.DataFrame -> to_dict('records')
      - pyspark.sql.DataFrame -> toLocalIterator() OR toPandas() (small)
      - list[dict]
      - list[tuple/list] (assumes CANON_COLS order)
    """
    if rows is None:
        return []

    # pandas DataFrame
    if hasattr(rows, "to_dict") and callable(getattr(rows, "to_dict")):
        try:
            recs = rows.to_dict("records")  # pandas
            if isinstance(recs, list):
                return [dict(r) for r in recs]
        except Exception:
            pass

    # pyspark DataFrame
    # heuristics: has .toLocalIterator and .columns and .schema
    if hasattr(rows, "toLocalIterator") and hasattr(rows, "columns"):
        cols = list(getattr(rows, "columns"))
        out: List[Dict[str, Any]] = []
        for r in rows.toLocalIterator():  # Spark Row objects
            d = r.asDict(recursive=True) if hasattr(r, "asDict") else {c: getattr(r, c, None) for c in cols}
            out.append(d)
        return out

    # list forms
    if isinstance(rows, list):
        if len(rows) == 0:
            return []
        if all(isinstance(r, dict) for r in rows):
            return [dict(r) for r in rows]
        if all(isinstance(r, (list, tuple)) for r in rows):
            out: List[Dict[str, Any]] = []
            for r in rows:
                d = {}
                for i, col in enumerate(CANON_COLS):
                    d[col] = r[i] if i < len(r) else None
                out.append(d)
            return out

    raise TypeError(
        "section_tables items must be a pandas DataFrame, Spark DataFrame, list[dict], or list[tuple/list]."
    )


def _write_table_sheet(wb: Workbook, title: str, rows: List[Dict[str, Any]]) -> None:
    ws = wb.create_sheet(_safe_sheet_title(title))

    # Header
    header_font = Font(bold=True)
    for col_idx, col in enumerate(CANON_COLS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col)
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # Data rows
    for r_idx, r in enumerate(rows, start=2):
        for c_idx, col in enumerate(CANON_COLS, start=1):
            ws.cell(row=r_idx, column=c_idx, value=r.get(col))

    # Freeze header
    ws.freeze_panes = "A2"

    # Add Excel Table (for filters/sort)
    max_row = max(2, 1 + len(rows))
    max_col = len(CANON_COLS)
    ref = f"A1:{get_column_letter(max_col)}{max_row}"
    table_name = "T" + "".join(ch for ch in _safe_sheet_title(title) if ch.isalnum())[:20]
    if not table_name or table_name == "T":
        table_name = "Table1"
    table = Table(displayName=table_name, ref=ref)
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium9",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )
    ws.add_table(table)

    # Column widths (nice-ish)
    widths = {
        "sort_key": 10,
        "KPI": 38,
        "P4WA": 16,
        "Yesterday": 16,
        "Delta": 14,
        "Call_Count_Delta": 18,
    }
    for i, col in enumerate(CANON_COLS, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(col, 16)

    # Align numeric columns
    for row in ws.iter_rows(min_row=2, min_col=1, max_col=max_col, max_row=max_row):
        # sort_key center
        row[0].alignment = Alignment(horizontal="center")
        # KPI left
        row[1].alignment = Alignment(horizontal="left")
        # numbers right
        for c in row[2:]:
            c.alignment = Alignment(horizontal="right")


def build_daily_kpis_workbook_bytes(*, section_names: List[str], section_tables: List[Any]) -> bytes:
    if len(section_names) != len(section_tables):
        raise ValueError(f"section_names length ({len(section_names)}) != section_tables length ({len(section_tables)})")

    wb = Workbook()
    # remove default sheet
    default = wb.active
    wb.remove(default)

    for name, table in zip(section_names, section_tables):
        rows = _coerce_rows_to_dicts(table)
        _write_table_sheet(wb, name, rows)

    bio = BytesIO()
    wb.save(bio)
    return bio.getvalue()


# -----------------------------
# End-to-end: DOCX + Excel upload
# -----------------------------
def upload_report_and_daily_kpis_xlsx(
    *,
    docx_path: str,
    channel_id: str,
    s1: Dict[str, Any],
    timeout_s: int = 30,
    env_file: str = ".env",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Uses:
      - s1['section_names'] (sheet names; e.g. bucket, site/serp, overall)
      - s1['section_tables'] (SQL output per section)

    Uploads:
      - DOCX with its current filename
      - Excel named exactly 'daily_kpis.xlsx'

    No Slack comments/initial messages for files.
    """
    p = Path(docx_path)
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f"docx_path not found: {docx_path!r}")

    section_names = s1.get("section_names") or []
    section_tables = s1.get("section_tables") or []
    if not isinstance(section_names, list) or not isinstance(section_tables, list):
        raise TypeError("s1['section_names'] and s1['section_tables'] must both be lists")

    xlsx_bytes = build_daily_kpis_workbook_bytes(
        section_names=section_names,
        section_tables=section_tables,
    )

    # Upload DOCX (keep name)
    docx_resp = _upload_bytes_via_external_flow(
        filename=p.name,
        content=p.read_bytes(),
        channel_id=channel_id.strip(),
        title=p.name,
        timeout_s=timeout_s,
        env_file=env_file,
    )

    # Upload Excel (fixed name)
    excel_resp = _upload_bytes_via_external_flow(
        filename="daily_kpis.xlsx",
        content=xlsx_bytes,
        channel_id=channel_id.strip(),
        title="daily_kpis.xlsx",
        timeout_s=timeout_s,
        env_file=env_file,
    )

    return docx_resp, excel_resp

# Cleanup

### Add Report to Knowledge

In [0]:
def save_final_report_to_knowledge_docx(
    *,
    final_report: Dict[str, Any],
    knowledge_dir: str | Path = "knowledge",
    filename: Optional[str] = None,  # e.g. "01_30_2026_report.docx"
    overwrite: bool = False,
) -> str:
    """
    Save finalize_report() output as a DOCX in <repo_root>/knowledge/previousReports
    (not relative to the current working directory).

    Uses final_report["combined_text"] as the document body.

    Returns the path to the DOCX written.
    """
    if not isinstance(final_report, dict):
        raise TypeError("final_report must be a dict (finalize_report() return value).")

    combined_text = final_report.get("combined_text")
    if not combined_text or not str(combined_text).strip():
        raise ValueError("final_report is missing non-empty 'combined_text'.")

    # ---- find repo root by walking upward until we see a "knowledge" dir or a marker file ----
    # Adjust markers if your repo uses something else (pyproject.toml, setup.cfg, etc.)
    start = Path.cwd().resolve()
    repo_root: Optional[Path] = None
    for p in [start] + list(start.parents):
        # prefer an actual "knowledge" folder if present
        if (p / "knowledge").exists():
            repo_root = p
            break
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            repo_root = p
            break

    # Fallback: if nothing found, default to cwd (won't crash, but may create shallow dirs)
    repo_root = repo_root or start

    # If we accidentally resolved to a "src" directory (common when running from inside src),
    # move up one level to the repository root so knowledge/ ends up at the true repo root.
    # Also, if the chosen repo_root doesn't contain "knowledge" but its parent does, prefer the parent.
    if repo_root.name == "src":
        repo_root = repo_root.parent
    elif not (repo_root / "knowledge").exists() and (repo_root.parent / "knowledge").exists():
        repo_root = repo_root.parent

    # Always write to <repo_root>/knowledge/previousReports
    reports_dir = (Path(repo_root) / Path(knowledge_dir) / "previousReports").resolve()
    reports_dir.mkdir(parents=True, exist_ok=True)

    if filename is None:
        date_key = time.strftime("%m_%d_%Y")
        filename = f"{date_key}_report.docx"

    if not filename.lower().endswith(".docx"):
        filename += ".docx"

    out_path = reports_dir / filename

    if out_path.exists() and not overwrite:
        raise FileExistsError(
            f"Report already exists: {out_path}. Pass overwrite=True to replace."
        )

    doc = Document()
    doc.add_heading("Daily KPI Report", level=1)

    for line in str(combined_text).splitlines():
        doc.add_paragraph("" if line.strip() == "" else line)

    doc.save(str(out_path))
    return str(out_path)


### Summarize Report & Save

In [0]:
def summarize_current_report_to_summaries(
    *,
    report_docx_path: str | Path,
    knowledge_dir: str | Path = "knowledge",
    model: str = "gpt-5-mini",
    max_report_chars: int = 60_000,
    overwrite: bool = False,
    output_filename: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Summarize ONE current report DOCX and save the summary as a DOCX in
    <repo_root>/knowledge/summaries.

    This version is defensive about the date regex (uses a local fallback if
    a module-level `_REPORT_DATE_RE` is not defined) and uses `search()` so the
    date can appear anywhere in the filename. It also guards against invalid
    parsed dates.
    """
    # Local imports to make the function self-contained / robust in different call sites
    import re
    import json
    from pathlib import Path
    from datetime import datetime
    from docx import Document
    from openai import OpenAI

    report_docx_path = Path(report_docx_path).resolve()
    if not report_docx_path.exists():
        raise FileNotFoundError(f"Report not found: {report_docx_path}")

    # ---- find repo root by walking upward from report path ----
    start_dir = report_docx_path.parent
    repo_root: Optional[Path] = None
    for p in [start_dir] + list(start_dir.parents):
        if (p / "knowledge").exists():
            repo_root = p
            break
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            repo_root = p
            break
    repo_root = repo_root or Path.cwd().resolve()

    # If we accidentally resolved to a "src" directory (common when running from inside src),
    # move up one level to the repository root so knowledge/ ends up at the true repo root.
    # Also prefer the parent if it contains "knowledge" even when repo_root doesn't.
    if repo_root.name == "src":
        repo_root = repo_root.parent
    elif not (repo_root / "knowledge").exists() and (repo_root.parent / "knowledge").exists():
        repo_root = repo_root.parent

    summaries_dir = (Path(repo_root) / Path(knowledge_dir) / "summaries").resolve()
    summaries_dir.mkdir(parents=True, exist_ok=True)

    # ---- Date extraction: use module-level regex if present, otherwise compile a local one ----
    # Expect named groups yyyy, mm, dd (the original code expects m["yyyy"], etc.)
    try:
        _regex = globals().get("_REPORT_DATE_RE")
        if _regex is None:
            raise NameError
        # ensure it's a compiled pattern
        if not hasattr(_regex, "search"):
            raise NameError
        REPORT_DATE_RE = _regex
    except Exception:
        REPORT_DATE_RE = re.compile(r"(?P<yyyy>\d{4})[_-](?P<mm>\d{2})[_-](?P<dd>\d{2})")

    # Parse date from filename (search anywhere in the name). Fallback to now when parse fails.
    m = REPORT_DATE_RE.search(report_docx_path.name)
    if m:
        try:
            dt = datetime(int(m["yyyy"]), int(m["mm"]), int(m["dd"]))
        except Exception:
            dt = datetime.now()
    else:
        dt = datetime.now()

    date_key = dt.strftime("%m_%d_%Y")
    day_of_week = dt.strftime("%A")

    # Read report text from DOCX
    doc_in = Document(str(report_docx_path))
    parts: List[str] = []
    for p in doc_in.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    report_text = "\n".join(parts)[:max_report_chars]

    # Build prompt locally (keep JSON braces escaped for .format safety if any formatting is used later)
    prompt = f"""
    You are summarizing a daily operations KPI report.

    Report date: {date_key}

    TASK
    A) EXECUTIVE SUMMARY
    - Write a 5–6 sentence executive summary capturing overall performance direction (better/worse/mixed),
    the biggest drivers, and any notable anomalies or consistency with prior days.
    - Do NOT include specific numbers.

    B) CALLOUTS
    - Provide 3–7 smaller trend callouts (step-level rates, queue/breakage, bot vs non-bot, funnel shifts, etc.).
    - Each callout must be ONE short sentence, no numbers.
    - Each callout must include an importance label: High | Medium | Low.

    OUTPUT FORMAT (STRICT JSON ONLY)
    {{
    "executive_summary": "<5-6 sentences>",
    "callouts": [
        {{"importance": "High", "text": "<callout sentence>"}},
        ...
    ]
    }}

    REPORT TEXT
    \"\"\"
    {report_text}
    \"\"\"
    """.strip()

    client = OpenAI()
    resp = client.responses.create(model=model, input=prompt)
    raw = (getattr(resp, "output_text", None) or "").strip()

    # Parse JSON (fallback: extract first JSON object)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        start_i = raw.find("{")
        end_i = raw.rfind("}")
        if start_i == -1 or end_i == -1 or end_i <= start_i:
            raise ValueError(f"Model did not return JSON. Output was:\n{raw}")
        data = json.loads(raw[start_i : end_i + 1])

    exec_summary = str(data.get("executive_summary", "")).strip()
    callouts = data.get("callouts") if isinstance(data.get("callouts"), list) else []

    # Normalize callouts
    cleaned_callouts: List[Dict[str, str]] = []
    for c in callouts:
        if not isinstance(c, dict):
            continue
        importance = str(c.get("importance", "")).strip().title()
        if importance not in {"High", "Medium", "Low"}:
            importance = "Medium"
        text = str(c.get("text", "")).strip()
        if text:
            cleaned_callouts.append({"importance": importance, "text": text})

    # Output filename
    if output_filename is None:
        output_filename = f"{date_key}_summary.docx"
    if not output_filename.lower().endswith(".docx"):
        output_filename += ".docx"

    summary_docx_path = (summaries_dir / output_filename).resolve()
    if summary_docx_path.exists() and not overwrite:
        raise FileExistsError(
            f"Summary already exists: {summary_docx_path}. Pass overwrite=True to replace."
        )

    # Write summary docx
    doc_out = Document()
    doc_out.add_heading(f"Summary - {date_key} ({day_of_week})", level=1)
    doc_out.add_paragraph(exec_summary if exec_summary else "(No executive summary returned.)")
    doc_out.add_heading("Callouts (with importance)", level=2)

    if cleaned_callouts:
        for c in cleaned_callouts:
            doc_out.add_paragraph(f"[{c['importance']}] {c['text']}", style="List Bullet")
    else:
        doc_out.add_paragraph("(No callouts returned.)")

    doc_out.save(str(summary_docx_path))

    return {
        "summary_docx_path": str(summary_docx_path),
        "executive_summary": exec_summary,
        "callouts": cleaned_callouts,
        "raw_model_output": raw,
    }


# Main Execution

In [0]:
from __future__ import annotations

import os
import time
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

from dotenv import load_dotenv
from tqdm import tqdm

def _stamp(msg: str, t0: Optional[float] = None) -> None:
    """
    Lightweight logger for pipeline steps.

    Args:
        msg: Message to print.
        t0: Optional start time (from time.time()) to print elapsed seconds.
    """
    # Toggle with LOG_STAMPS=0/false/no to silence
    val = os.getenv("LOG_STAMPS", "1").strip().lower()
    if val in {"0", "false", "no", "off"}:
        return

    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    if t0 is None:
        print(f"[{ts}] {msg}")
    else:
        elapsed = time.time() - float(t0)
        print(f"[{ts}] {msg} (+{elapsed:0.2f}s)")


# ============================================================
# 1) Start -> JSON creation
# ============================================================
def step1_build_kpi_json(
    *,
    company_id: int = 25,
    call_direction: str = "INBOUND",
    restrict_to_yday_and_prior4: bool = True,
    allowed_buckets: Optional[Set[str]] = None,
    site_serp_values: Sequence[str] = ("Site", "SERP"),
    # compute_kpis params
    create_temp_view_for_kpis: bool = True,
    debug_print_sql: bool = False,
    # filter params
    min_metric_current_calls: int = 5,
    min_abs_delta_pct: float = 0.01,
    min_breakdown_current_calls: int = 5,
    breakdown_top_n: int = 5,
) -> Dict[str, Any]:
    """
    Builds KPI tables (overall + buckets + site/serp) -> kpi_json_full -> kpi_json_filtered.
    Returns everything needed for downstream steps.
    """
    if allowed_buckets is None:
        allowed_buckets = {
            "Natural",
            "Brand-Partner",
            "Generic",
            "Aggregator",
            "Competitor",
            "Utility",
            "PMax",
            "NRG",
            "Other Bucket",
        }

    t0 = time.time()
    _stamp("STEP 1 START: KPI build -> JSON", t0)

    section_names: List[str] = []
    section_tables: List[Any] = []

    # OVERALL
    _stamp("STEP 1A: overall", t0)
    calls_df_overall = get_data(
        company_id=company_id,
        call_direction=call_direction,
        restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
        create_temp_view=False,
        marketing_buckets=None,
        site_serp=None,
    )
    kpi_df_overall = compute_kpis(
         calls_df=calls_df_overall,
         source_view="calls_full_debug_overall",
         create_temp_view=create_temp_view_for_kpis,
         debug_print_sql=debug_print_sql,
     )

    mixes = compute_mix_shifts(calls_df_overall, create_temp_view=True)

    section_names.append("overall")
    section_tables.append(kpi_df_overall)

    # BUCKETS
    buckets_sorted = sorted(list(allowed_buckets))
    _stamp(f"STEP 1B: buckets | n={len(buckets_sorted)}", t0)
    for bucket in tqdm(buckets_sorted, desc="Marketing buckets"):
        bucket_snake = _to_snake(bucket)

        calls_df_bucket = get_data(
            company_id=company_id,
            call_direction=call_direction,
            restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
            create_temp_view=False,
            marketing_buckets=bucket,
            site_serp=None,
        )
        kpi_df_bucket = compute_kpis(
            calls_df=calls_df_bucket,
            source_view=f"calls_full_debug_bucket_{bucket_snake}",
            create_temp_view=create_temp_view_for_kpis,
            debug_print_sql=debug_print_sql,
        )

        section_names.append(f"bucket_{bucket_snake}")
        section_tables.append(kpi_df_bucket)

    # SITE/SERP
    _stamp(f"STEP 1C: site/serp | values={list(site_serp_values)}", t0)
    for ss in tqdm(site_serp_values, desc="Site vs SERP"):
        ss_snake = _to_snake(ss)

        calls_df_ss = get_data(
            company_id=company_id,
            call_direction=call_direction,
            restrict_to_yday_and_prior4=restrict_to_yday_and_prior4,
            create_temp_view=False,
            marketing_buckets=None,
            site_serp=ss,
        )
        kpi_df_ss = compute_kpis(
            calls_df=calls_df_ss,
            source_view=f"calls_full_debug_{ss_snake}",
            create_temp_view=create_temp_view_for_kpis,
            debug_print_sql=debug_print_sql,
        )

        section_names.append(ss_snake)
        section_tables.append(kpi_df_ss)

    # JSON build + filter
    _stamp(f"STEP 1D: JSON build | sections={len(section_names)}", t0)
    kpi_json_full = kpi_tables_to_json(names=section_names, tables=section_tables, mixes=mixes)

    _stamp("STEP 1E: filter_and_compact_kpi_json()", t0)
    kpi_json_filtered = filter_and_compact_kpi_json(
        kpi_json_full,
        min_metric_current_calls=min_metric_current_calls,
        min_abs_delta_pct=min_abs_delta_pct,
        min_breakdown_current_calls=min_breakdown_current_calls,
        breakdown_top_n=breakdown_top_n,
    )

    _stamp("STEP 1 DONE", t0)
    return {
        "kpi_json_full": kpi_json_full,
        "kpi_json_filtered": kpi_json_filtered,
        "section_names": section_names,
        "section_tables": section_tables,
    }

# robust step2_build_context: searches upward for .env and loads it
def _find_upward_env(start_path: Optional[str] = None, filename: str = ".env", max_levels: int = 10) -> Optional[str]:
    """
    Search upward from start_path (or cwd) up to max_levels for filename.
    Returns the absolute path to the first match, or None.
    """
    from pathlib import Path

    p = Path(start_path) if start_path else Path.cwd()
    p = p.resolve()
    for _ in range(max_levels + 1):
        candidate = p / filename
        if candidate.exists() and candidate.is_file():
            return str(candidate)
        if p.parent == p:
            break
        p = p.parent
    return None


def step2_build_context(
    *,
    kpi_json_filtered: Any,
    knowledge_dir: str = "knowledge",
    lookback_summaries: int = 3,
    n_queries: int = 6,
    model: str = "gpt-5-mini",
    max_results_per_query: int = 5,
    base_business_context: Optional[str] = None,
    env_file: str = ".env",  # can be relative or absolute
    vector_store_env_key: str = "OPENAI_VECTOR_STORE_ID",
) -> Dict[str, Any]:
    """
    Uses filtered KPI JSON to generate query plan and retrieve context blocks.
    Robustly loads the env file by searching upwards from cwd if needed.
    """
    t0 = time.time()
    _stamp("STEP 2 START: JSON -> context", t0)

    # Attempt 1: if env_file is an absolute or explicit path and exists, use it
    env_path = None
    if env_file:
        p = Path(env_file)
        if p.is_absolute() and p.exists():
            env_path = str(p)
        elif p.exists():
            env_path = str(p.resolve())

    # Attempt 2: search upward from cwd for the filename (only if env_path not already found)
    if not env_path:
        found = _find_upward_env(start_path=None, filename=env_file or ".env", max_levels=20)
        if found:
            env_path = found

    # Debug info: show where we will try to load from
    print(f"[DEBUG] resolved env_path: {env_path!r} (cwd={Path.cwd()})")

    loaded = False
    if env_path:
        # load_dotenv returns True if a file was found and parsed
        try:
            loaded = load_dotenv(dotenv_path=env_path)
            print(f"[DEBUG] load_dotenv('{env_path}') returned: {loaded}")
        except Exception as e:
            print(f"[DEBUG] load_dotenv raised: {e}")

    # Fallback: try plain load_dotenv() (searches common locations)
    if not loaded:
        try:
            loaded = load_dotenv()
            print(f"[DEBUG] load_dotenv() fallback returned: {loaded}")
        except Exception as e:
            print(f"[DEBUG] load_dotenv() fallback raised: {e}")

    # Final fallback: manual parse if file exists but wasn't loaded
    if not loaded and env_path and os.path.exists(env_path):
        print(f"[DEBUG] manual fallback parse for {env_path!r}")
        try:
            with open(env_path, "r", encoding="utf-8") as f:
                for raw in f:
                    line = raw.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    k, v = line.split("=", 1)
                    k = k.strip()
                    v = v.strip().strip('"').strip("'")
                    if k:
                        os.environ.setdefault(k, v)
            loaded = True
            print("[DEBUG] manual parse complete")
        except Exception as e:
            print(f"[DEBUG] manual parse failed: {e}")

    vector_store_id = os.getenv(vector_store_env_key)
    print(f"[DEBUG] {vector_store_env_key} present: {bool(vector_store_id)}")

    if not vector_store_id:
        raise ValueError(f"{vector_store_env_key} is not set in the environment/.env (tried: {env_path!r})")

    # --- existing logic continues ---
    weekly_context = get_weekly_context(knowledge_dir, lookback_summaries=lookback_summaries)

    query_plan = generate_vector_store_queries(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        n_queries=n_queries,
        model=model,
        base_business_context=base_business_context,
    )

    retrieved_blocks = retrieve_vector_store_blocks(
        vector_store_id=vector_store_id,
        query_plan=query_plan,
        max_results_per_query=max_results_per_query,
        save_path=None,
    )

    context_vars = {
        "kpi_json_filtered": kpi_json_filtered,
        "weekly_context": weekly_context,
        "query_plan": query_plan,
        "retrieved_blocks": retrieved_blocks,
        "vector_store_id": vector_store_id,
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }

    _stamp("STEP 2 DONE", t0)
    return context_vars


def step3_generate_outputs(
    *,
    kpi_json_filtered: Any,
    pacing_payload: Dict[str, Any],  # <-- ADD: dict from your pacing query runner (MTD pacing vs internal plan)
    model: str = "gpt-5",
    # pacing params
    # (keep these for backward compatibility, but they won't be used once alarms is removed)
    min_alerts: int = 3,
    min_monitor_items: int = 4,
    alarms_debug: bool = True,
    context_vars
) -> Dict[str, Any]:
    """
    Builds shared context once, then runs the OpenAI generation calls
    using that same context.

    Now generates:
      1) Executive summary
      2) Step-level drivers
      3) MTD pacing narrative (replaces out-of-scope analyses & alarms)
    """
    t0 = time.time()
    _stamp("STEP 3 START: OpenAI generation", t0)

    weekly_context = context_vars.get("weekly_context")
    retrieved_blocks = context_vars.get("retrieved_blocks")

    # --------------------------------------------------
    # 2) Executive summary
    # --------------------------------------------------
    summary_text = generate_executive_summary_for_report_text(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        retrieved_blocks=retrieved_blocks,
        model=model,
    )

    # --------------------------------------------------
    # 3) Step-level drivers
    # --------------------------------------------------
    drivers_payload = generate_step_level_drivers(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        retrieved_blocks=retrieved_blocks,
        min_drivers=3,
        model=model,
    )

    # --------------------------------------------------
    # 4) Month-to-date pacing narrative (REPLACES alarms)
    # --------------------------------------------------
    pacing_result = generate_mtd_pacing_narrative(
        kpi_json=kpi_json_filtered,
        weekly_context=weekly_context,
        retrieved_blocks=retrieved_blocks,
        pacing_payload=pacing_payload, 
        model=model,
        debug=alarms_debug,  
    )

    _stamp("STEP 3 DONE", t0)

    return {
        "summary_text": summary_text,
        "drivers_payload": drivers_payload,
        "pacing_result": pacing_result
    }


# ============================================================
# 4) Format + Save DOCX + Slack
# ============================================================
from datetime import datetime
from typing import Any, Dict, Optional

def step4_finalize_and_send_to_slack(
    *,
    summary_text: str,
    drivers_payload: Dict[str, Any],
    pacing_result: Any,  # <-- CHANGED (replaces alarms_result)
    # reporting / limits
    max_driver_items: int = 10,
    # Slack params
    env_file: str = ".env",
    webhook_env_key: str = "SLACK_WEBHOOK_URL",
    slack_username: Optional[str] = None,
    slack_icon_emoji: Optional[str] = None,
    # Knowledge / DOCX save params
    knowledge_dir: str = "knowledge",
    overwrite_report_docx: bool = True,
) -> Dict[str, Any]:
    """
    Finalizes report formatting, saves the report DOCX, and posts to Slack.

    IMPORTANT:
    - We save the DOCX *before* posting to Slack so the Slack function can include a link.
    - The returned `final_report` dict is augmented with `docx_path` and `report_docx_path`
      for downstream use.
    """
    t0 = time.time()
    _stamp("STEP 4 START: finalize + save DOCX + Slack", t0)

    # --------------------------------------------------
    # 1) Build the final report text payload (UPDATED)
    # --------------------------------------------------
    final_report = finalize_report(
        summary=summary_text,
        drivers_payload=drivers_payload,
        pacing_result=pacing_result,  # <-- UPDATED
        max_driver_items=max_driver_items,
    )

    # --------------------------------------------------
    # 2) Save report DOCX FIRST (so Slack can link it)
    # --------------------------------------------------
    date_key = datetime.now().strftime("%m_%d_%Y")
    report_docx_path = save_final_report_to_knowledge_docx(
        final_report=final_report,
        knowledge_dir=knowledge_dir,
        filename=f"{date_key}_report.docx",
        overwrite=overwrite_report_docx,
    )
    print("Saved report DOCX:", report_docx_path)

    # --------------------------------------------------
    # 3) Summarize THAT file (don’t hardcode the path)
    # --------------------------------------------------
    summary_out = summarize_current_report_to_summaries(
        report_docx_path=report_docx_path,
        knowledge_dir=knowledge_dir,
        model="gpt-5",
        overwrite=True,
    )

    # Add keys the Slack poster knows how to detect
    final_report["docx_path"] = report_docx_path
    final_report["report_docx_path"] = report_docx_path  # optional extra for clarity

    # (Optional) persist summarizer output for debugging / downstream
    final_report["report_summary_out"] = summary_out

    # --------------------------------------------------
    # 4) Post to Slack (now includes link if your post function supports it)
    # --------------------------------------------------
    docx_resp, excel_resp = upload_report_and_daily_kpis_xlsx(
        docx_path=report_docx_path,
        channel_id="C0ACY9XB82E",
        s1=s1,
    )


    _stamp("STEP 4 DONE", t0)
    return final_report

s1=None
ctx=None
pacing_payload=None
gen=None
final_report=None
from datetime import datetime

# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    s1 = step1_build_kpi_json()
    # Build context once (as you already do)
    ctx = step2_build_context(kpi_json_filtered=s1["kpi_json_full"])

    # Use whatever warehouse client you have; this assumes you implemented `wh`.
    pacing_payload = fetch_tx_mtd_pacing(            
        include_social="Yes", 
    )

    gen = step3_generate_outputs(
        kpi_json_filtered=s1["kpi_json_full"],
        pacing_payload=pacing_payload, 
        model="gpt-5",
        context_vars=ctx
    )

    final_report = step4_finalize_and_send_to_slack(
        summary_text=gen["summary_text"],
        drivers_payload=gen["drivers_payload"],
        pacing_result=gen["pacing_result"], 
    )

    summary_out = final_report.get("report_summary_out") or {}
    if summary_out.get("summary_docx_path"):
        print("Saved summary DOCX:", summary_out["summary_docx_path"])

    print("\n--- Final combined report (truncated) ---\n")
    print(final_report["combined_text"][:4000])